In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path("/home/mark/dabtroll")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOGS_DIR = PROJECT_ROOT / "data" / "logs"
MAX_NODES = 4

import scripts.notebook_qwen_compare_utils as nqcu
nqcu = importlib.reload(nqcu)

# Build the same standardized episode dataframe used in dataset_dev.
episode_df = nqcu.build_all_episode_workbook_rows(
    logs_dir=LOGS_DIR,
    max_nodes=MAX_NODES,
    export_csv=False,
)

# Match the analysis scope used in VIBTRAVE-style comparisons.
episode_df["episode_index"] = pd.to_numeric(episode_df["episode_index"], errors="coerce")
episode_df_2_21 = episode_df[episode_df["episode_index"].between(2, 21, inclusive="both")].copy()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

display(Markdown("## Loaded Episode Data"))
print("episode_df shape:", episode_df.shape)
print("episode_df_2_21 shape:", episode_df_2_21.shape)
display(
    episode_df_2_21[[
        "scenario_id",
        "episode_dir",
        "episode_index",
        "mission_name",
        "success",
        "bt_action_nodes",
    ]].head(12)
)

## Benchmark vs Simulator and BT All-Nodes-Complete Statistical Summary

This section computes, per scenario and per model:
- Simulator success rate.
- BT all-action-nodes-complete rate (proxy for predicted success).
- Delta versus posted GR1 benchmark rate.
- 95% confidence intervals and comparison flags.

It also computes overall model diagnostics against simulator success (TP/FP/TN/FN, precision/recall/specificity/accuracy/F1) and pairwise model comparisons.

In [ ]:
import math

if "episode_df_2_21" not in globals() or episode_df_2_21.empty:
    raise ValueError("episode_df_2_21 is missing or empty. Run Cell 1 first.")

df = episode_df_2_21.copy()

required_base_cols = ["scenario_id", "episode_dir", "bt_action_nodes", "success"]
missing_base = [c for c in required_base_cols if c not in df.columns]
if missing_base:
    raise ValueError(f"episode_df is missing required columns: {missing_base}")


def _to_success_bool(v: object) -> object:
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return np.nan


def _safe_int(v: object, default: int = 0) -> int:
    try:
        if pd.isna(v):
            return default
        return int(v)
    except Exception:
        return default


def _is_time_filled(v: object) -> bool:
    if pd.isna(v):
        return False
    return str(v).strip() != ""


def _all_nodes_complete(row: pd.Series, completion_suffix: str) -> tuple[int, int, bool]:
    required = max(0, _safe_int(row.get("bt_action_nodes", 0), default=0))
    if required == 0:
        return 0, 0, False

    filled = 0
    for i in range(1, required + 1):
        c = f"node_{i}_{completion_suffix}"
        filled += int(_is_time_filled(row.get(c, np.nan)))

    return required, filled, bool(filled == required)


def _binom_ci_95(successes: int, n: int) -> tuple[float, float]:
    if n <= 0:
        return (np.nan, np.nan)
    p = successes / n
    se = np.sqrt(p * (1.0 - p) / n)
    lo = max(0.0, p - 1.96 * se)
    hi = min(1.0, p + 1.96 * se)
    return (lo, hi)


def _task_key(s: object) -> str:
    txt = "" if pd.isna(s) else str(s)
    txt = txt.split("/")[-1]
    txt = txt.replace("_GR1ArmsAndWaistFourierHands_Env", "")
    return txt.lower()


MODEL_SPECS = [
    {"key": "qwen3", "label": "qwen3", "suffix": "model_completion_time"},
    {"key": "qwen35", "label": "qwen3.5", "suffix": "qwen3_5_model_completion_time"},
    {"key": "qwen35_mod1", "label": "qwen3.5 mod1", "suffix": "qwen3_5_mod1_completion_time"},
]

df["sim_success_bool"] = df["success"].map(_to_success_bool)

for spec in MODEL_SPECS:
    triplets = df.apply(lambda r, s=spec["suffix"]: _all_nodes_complete(r, s), axis=1)
    df[f"{spec['key']}_required_slots"] = triplets.map(lambda t: t[0])
    df[f"{spec['key']}_filled_slots"] = triplets.map(lambda t: t[1])
    df[f"{spec['key']}_all_nodes_complete"] = triplets.map(lambda t: t[2])

BENCHMARK_DF = pd.DataFrame([
    {"task": "gr1_unified/PnPBottleToCabinetClose_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 51.5},
    {"task": "gr1_unified/PnPCanToDrawerClose_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 13.0},
    {"task": "gr1_unified/PnPCupToDrawerClose_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 8.5},
    {"task": "gr1_unified/PnPMilkToMicrowaveClose_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 14.0},
    {"task": "gr1_unified/PnPPotatoToMicrowaveClose_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 41.5},
    {"task": "gr1_unified/PnPWineToCabinetClose_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 16.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromCuttingboardToBasketSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 58.0},
    {"task": "gr1_unified/PosttrainPnPNovelFromCuttingboardToCardboardboxSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 46.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromCuttingboardToPanSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 68.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromCuttingboardToPotSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 65.0},
    {"task": "gr1_unified/PosttrainPnPNovelFromCuttingboardToTieredbasketSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 46.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlacematToBasketSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 58.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlacematToBowlSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 57.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlacematToPlateSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 63.0},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlacematToTieredshelfSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 28.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlateToBowlSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 57.0},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlateToCardboardboxSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 43.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlateToPanSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 51.0},
    {"task": "gr1_unified/PosttrainPnPNovelFromPlateToPlateSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 78.7},
    {"task": "gr1_unified/PosttrainPnPNovelFromTrayToCardboardboxSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 51.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromTrayToPlateSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 71.0},
    {"task": "gr1_unified/PosttrainPnPNovelFromTrayToPotSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 64.5},
    {"task": "gr1_unified/PosttrainPnPNovelFromTrayToTieredbasketSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 57.0},
    {"task": "gr1_unified/PosttrainPnPNovelFromTrayToTieredshelfSplitA_GR1ArmsAndWaistFourierHands_Env", "benchmark_success_rate_pct": 31.5},
])
BENCHMARK_DF["task_key"] = BENCHMARK_DF["task"].map(_task_key)

scenario_map = pd.DataFrame({"scenario_id": sorted(df["scenario_id"].dropna().astype(str).unique())})
scenario_map["task_key"] = scenario_map["scenario_id"].map(_task_key)
scenario_map = scenario_map.merge(
    BENCHMARK_DF[["task_key", "benchmark_success_rate_pct"]],
    on="task_key",
    how="left",
)

metric_defs = [{"metric": "sim_success", "label": "simulator", "col": "sim_success_bool"}]
for spec in MODEL_SPECS:
    metric_defs.append({
        "metric": f"{spec['key']}_all_nodes_complete",
        "label": spec["label"],
        "col": f"{spec['key']}_all_nodes_complete",
    })

scenario_records = []
for sc, g in df.groupby("scenario_id", dropna=False):
    n = int(len(g))
    for md in metric_defs:
        s = pd.Series(g[md["col"]]).astype(float)
        successes = int(np.nansum(s))
        rate = 100.0 * successes / n if n > 0 else np.nan
        ci_lo, ci_hi = _binom_ci_95(successes, n)
        scenario_records.append({
            "scenario_id": sc,
            "metric": md["metric"],
            "model": md["label"],
            "episodes": n,
            "successes": successes,
            "rate_pct": round(rate, 1) if pd.notna(rate) else np.nan,
            "ci95_lo_pct": round(100.0 * ci_lo, 1) if pd.notna(ci_lo) else np.nan,
            "ci95_hi_pct": round(100.0 * ci_hi, 1) if pd.notna(ci_hi) else np.nan,
        })

SCENARIO_RATE_TABLE = pd.DataFrame(scenario_records).merge(
    scenario_map[["scenario_id", "benchmark_success_rate_pct"]],
    on="scenario_id",
    how="left",
)
SCENARIO_RATE_TABLE["delta_vs_benchmark_pct"] = (
    SCENARIO_RATE_TABLE["rate_pct"] - SCENARIO_RATE_TABLE["benchmark_success_rate_pct"]
).round(1)
SCENARIO_RATE_TABLE["benchmark_within_ci95"] = (
    SCENARIO_RATE_TABLE["benchmark_success_rate_pct"].ge(SCENARIO_RATE_TABLE["ci95_lo_pct"])
) & (
    SCENARIO_RATE_TABLE["benchmark_success_rate_pct"].le(SCENARIO_RATE_TABLE["ci95_hi_pct"])
)
SCENARIO_RATE_TABLE["outside_95moe_proxy"] = (
    SCENARIO_RATE_TABLE["delta_vs_benchmark_pct"].abs()
    > (SCENARIO_RATE_TABLE["ci95_hi_pct"] - SCENARIO_RATE_TABLE["rate_pct"]).abs()
)

valid = df[df["sim_success_bool"].isin([True, False])].copy()
overall_rows = []
for spec in MODEL_SPECS:
    mk = spec["key"]
    pred_col = f"{mk}_all_nodes_complete"
    if pred_col not in valid.columns:
        continue

    actual = valid["sim_success_bool"].astype(bool)
    pred = valid[pred_col].astype(bool)

    tp = int((actual & pred).sum())
    fp = int((~actual & pred).sum())
    tn = int((~actual & ~pred).sum())
    fn = int((actual & ~pred).sum())
    n = int(len(valid))

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    accuracy = (tp + tn) / n if n else np.nan
    f1 = (2 * precision * recall / (precision + recall)) if (pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0) else np.nan

    b = fn
    c = fp
    if (b + c) > 0:
        chi2 = ((abs(b - c) - 1) ** 2) / (b + c)
        p_mcnemar = float(math.erfc(math.sqrt(chi2 / 2.0)))
    else:
        chi2 = np.nan
        p_mcnemar = np.nan

    overall_rows.append({
        "model": spec["label"],
        "episodes_eval": n,
        "pred_success_rate_pct": round(100.0 * pred.mean(), 1),
        "sim_success_rate_pct": round(100.0 * actual.mean(), 1),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "precision": round(precision, 4) if pd.notna(precision) else np.nan,
        "recall": round(recall, 4) if pd.notna(recall) else np.nan,
        "specificity": round(specificity, 4) if pd.notna(specificity) else np.nan,
        "accuracy": round(accuracy, 4) if pd.notna(accuracy) else np.nan,
        "f1": round(f1, 4) if pd.notna(f1) else np.nan,
        "mcnemar_chi2_cc": round(chi2, 4) if pd.notna(chi2) else np.nan,
        "mcnemar_p_approx": round(p_mcnemar, 6) if pd.notna(p_mcnemar) else np.nan,
    })

OVERALL_MODEL_STATS = pd.DataFrame(overall_rows)

pairwise_rows = []
for i, a in enumerate(MODEL_SPECS):
    for b in MODEL_SPECS[i + 1 :]:
        col_a = f"{a['key']}_all_nodes_complete"
        col_b = f"{b['key']}_all_nodes_complete"
        if col_a not in valid.columns or col_b not in valid.columns:
            continue

        v = valid[[col_a, col_b]].dropna()
        if v.empty:
            continue

        rate_a = 100.0 * v[col_a].astype(bool).mean()
        rate_b = 100.0 * v[col_b].astype(bool).mean()
        diff = rate_a - rate_b

        a_only = int((v[col_a].astype(bool) & ~v[col_b].astype(bool)).sum())
        b_only = int((~v[col_a].astype(bool) & v[col_b].astype(bool)).sum())
        if (a_only + b_only) > 0:
            chi2_ab = ((abs(a_only - b_only) - 1) ** 2) / (a_only + b_only)
            p_ab = float(math.erfc(math.sqrt(chi2_ab / 2.0)))
        else:
            chi2_ab = np.nan
            p_ab = np.nan

        pairwise_rows.append({
            "model_a": a["label"],
            "model_b": b["label"],
            "episodes_eval": int(len(v)),
            "rate_a_pct": round(rate_a, 1),
            "rate_b_pct": round(rate_b, 1),
            "delta_a_minus_b_pct": round(diff, 1),
            "discordant_a_only": a_only,
            "discordant_b_only": b_only,
            "mcnemar_chi2_cc": round(chi2_ab, 4) if pd.notna(chi2_ab) else np.nan,
            "mcnemar_p_approx": round(p_ab, 6) if pd.notna(p_ab) else np.nan,
        })

PAIRWISE_MODEL_COMPARE = pd.DataFrame(pairwise_rows)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(Markdown("### Scenario-Level Summary: Simulator and BT All-Nodes-Complete vs Benchmark"))
display(
    SCENARIO_RATE_TABLE.sort_values(["scenario_id", "metric"]).reset_index(drop=True)[
        [
            "scenario_id",
            "model",
            "metric",
            "episodes",
            "successes",
            "rate_pct",
            "ci95_lo_pct",
            "ci95_hi_pct",
            "benchmark_success_rate_pct",
            "delta_vs_benchmark_pct",
            "benchmark_within_ci95",
            "outside_95moe_proxy",
        ]
    ]
)

display(Markdown("### Overall Model Diagnostics vs Simulator Success"))
display(OVERALL_MODEL_STATS.sort_values("model").reset_index(drop=True))

display(Markdown("### Pairwise Model Comparison (BT All-Nodes-Complete Rate)"))
display(PAIRWISE_MODEL_COMPARE.sort_values(["model_a", "model_b"]).reset_index(drop=True))

if not SCENARIO_RATE_TABLE.empty:
    matched = SCENARIO_RATE_TABLE[SCENARIO_RATE_TABLE["benchmark_success_rate_pct"].notna()].copy()
    outside = int(matched["outside_95moe_proxy"].fillna(False).sum())
    within = int(matched["benchmark_within_ci95"].fillna(False).sum())
    display(Markdown(
        "### Quick Interpretation\n"
        f"- Scenario-model rows with benchmark mapping: **{len(matched)}**.\n"
        f"- Rows where benchmark is inside 95% CI of observed rate: **{within}/{len(matched) if len(matched) else 0}**.\n"
        f"- Rows outside 95% margin-of-error proxy: **{outside}/{len(matched) if len(matched) else 0}**.\n"
        "- Use OVERALL_MODEL_STATS and PAIRWISE_MODEL_COMPARE for model-level statistical comparisons."
    ))

## Per-Scenario Tables (Short Scenario Names)

The next cell provides per-scenario versions of the summary tables using short scenario labels:
- Can to drawer close
- Cuttingboard to basket
- Plate to plate

Column guide for Scenario-Level Rate Summary:
- short_scenario: Human-readable scenario label.
- model: simulator, qwen3, qwen3.5, or qwen3.5 mod1.
- metric: `sim_success` for simulator outcome or `*_all_nodes_complete` for BT all-node completion.
- episodes: Number of episodes used for that scenario and metric.
- successes: Count of episodes marked successful by that metric.
- rate_pct: Success percentage (`successes / episodes * 100`).
- ci95_lo_pct / ci95_hi_pct: 95% normal-approximation confidence interval for the rate.
- benchmark_success_rate_pct: Posted GR1 benchmark success percentage for matched scenario.
- delta_vs_benchmark_pct: Observed rate minus benchmark (percentage points).
- benchmark_within_ci95: True if benchmark falls within the observed 95% CI.
- outside_95moe_proxy: True if absolute delta exceeds CI half-width proxy.

Column guide for Per-Scenario Model Diagnostics vs Simulator:
- tp: Model predicts success and simulator is successful.
- fp: Model predicts success but simulator fails.
- tn: Model predicts failure and simulator fails.
- fn: Model predicts failure but simulator succeeds.
- precision: `tp / (tp + fp)`; among model-predicted successes, how many are true.
- recall: `tp / (tp + fn)`; among simulator successes, how many model captures.
- specificity: `tn / (tn + fp)`; among simulator failures, how many model correctly marks.
- accuracy: `(tp + tn) / N` across all evaluated episodes.
- f1: Harmonic mean of precision and recall.
- mcnemar_chi2_cc / mcnemar_p_approx: Paired disagreement test between model and simulator labels.

Column guide for Per-Scenario Pairwise Model Comparison:
- model_a, model_b: Pair of models being compared.
- rate_a_pct, rate_b_pct: BT all-node-complete success rates per model.
- delta_a_minus_b_pct: Difference in rates (A minus B, percentage points).
- discordant_a_only / discordant_b_only: Episodes where only one model predicts success.
- mcnemar_chi2_cc / mcnemar_p_approx: Paired disagreement test between model A and model B predictions.

In [ ]:
if "SCENARIO_RATE_TABLE" not in globals() or SCENARIO_RATE_TABLE.empty:
    raise ValueError("SCENARIO_RATE_TABLE is missing. Run Cell 3 first.")
if "df" not in globals() or df.empty:
    raise ValueError("Working dataframe `df` is missing. Run Cell 3 first.")
if "MODEL_SPECS" not in globals() or len(MODEL_SPECS) == 0:
    raise ValueError("MODEL_SPECS is missing. Run Cell 3 first.")

# Short display names requested.
SHORT_SCENARIO_MAP = {
    "gr1_unified/PnPCanToDrawerClose_GR1ArmsAndWaistFourierHands_Env": "Can to drawer close",
    "gr1_unified/PosttrainPnPNovelFromCuttingboardToBasketSplitA_GR1ArmsAndWaistFourierHands_Env": "Cuttingboard to basket",
    "gr1_unified/PosttrainPnPNovelFromPlateToPlateSplitA_GR1ArmsAndWaistFourierHands_Env": "Plate to plate",
}
SCENARIO_ORDER = [
    "Can to drawer close",
    "Cuttingboard to basket",
    "Plate to plate",
]

# Benchmark lookup from the earlier summary cell (same source used in VIBTRAVE notebook logic).
benchmark_by_scenario = (
    SCENARIO_RATE_TABLE[["scenario_id", "benchmark_success_rate_pct"]]
    .drop_duplicates(subset=["scenario_id"])
    .set_index("scenario_id")["benchmark_success_rate_pct"]
    .to_dict()
)

# 1) Per-scenario rate summary with short names.
scenario_rates_short = SCENARIO_RATE_TABLE.copy()
scenario_rates_short = scenario_rates_short[scenario_rates_short["scenario_id"].isin(SHORT_SCENARIO_MAP.keys())].copy()
scenario_rates_short["short_scenario"] = scenario_rates_short["scenario_id"].map(SHORT_SCENARIO_MAP)
scenario_rates_short["short_scenario"] = pd.Categorical(
    scenario_rates_short["short_scenario"],
    categories=SCENARIO_ORDER,
    ordered=True,
)
SCENARIO_RATE_TABLE_SHORT = scenario_rates_short.sort_values(["short_scenario", "model", "metric"]).reset_index(drop=True)

# 2) Per-scenario model diagnostics vs simulator.
valid_local = df[df["sim_success_bool"].isin([True, False])].copy()
per_scenario_rows = []
for scenario_id, g in valid_local.groupby("scenario_id", dropna=False):
    if scenario_id not in SHORT_SCENARIO_MAP:
        continue
    actual = g["sim_success_bool"].astype(bool)
    n = int(len(g))

    for spec in MODEL_SPECS:
        mk = spec["key"]
        pred_col = f"{mk}_all_nodes_complete"
        if pred_col not in g.columns:
            continue
        pred = g[pred_col].astype(bool)

        tp = int((actual & pred).sum())
        fp = int((~actual & pred).sum())
        tn = int((~actual & ~pred).sum())
        fn = int((actual & ~pred).sum())

        precision = tp / (tp + fp) if (tp + fp) else np.nan
        recall = tp / (tp + fn) if (tp + fn) else np.nan
        specificity = tn / (tn + fp) if (tn + fp) else np.nan
        accuracy = (tp + tn) / n if n else np.nan
        f1 = (2 * precision * recall / (precision + recall)) if (pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0) else np.nan

        b = fn
        c = fp
        if (b + c) > 0:
            chi2 = ((abs(b - c) - 1) ** 2) / (b + c)
            p_mcnemar = float(math.erfc(math.sqrt(chi2 / 2.0)))
        else:
            chi2 = np.nan
            p_mcnemar = np.nan

        per_scenario_rows.append({
            "scenario_id": scenario_id,
            "short_scenario": SHORT_SCENARIO_MAP[scenario_id],
            "model": spec["label"],
            "episodes_eval": n,
            "pred_success_rate_pct": round(100.0 * pred.mean(), 1),
            "sim_success_rate_pct": round(100.0 * actual.mean(), 1),
            "benchmark_success_rate_pct": benchmark_by_scenario.get(scenario_id, np.nan),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
            "precision": round(precision, 4) if pd.notna(precision) else np.nan,
            "recall": round(recall, 4) if pd.notna(recall) else np.nan,
            "specificity": round(specificity, 4) if pd.notna(specificity) else np.nan,
            "accuracy": round(accuracy, 4) if pd.notna(accuracy) else np.nan,
            "f1": round(f1, 4) if pd.notna(f1) else np.nan,
            "mcnemar_chi2_cc": round(chi2, 4) if pd.notna(chi2) else np.nan,
            "mcnemar_p_approx": round(p_mcnemar, 6) if pd.notna(p_mcnemar) else np.nan,
        })

SCENARIO_MODEL_STATS = pd.DataFrame(per_scenario_rows)
if not SCENARIO_MODEL_STATS.empty:
    SCENARIO_MODEL_STATS["short_scenario"] = pd.Categorical(
        SCENARIO_MODEL_STATS["short_scenario"],
        categories=SCENARIO_ORDER,
        ordered=True,
    )
    SCENARIO_MODEL_STATS = SCENARIO_MODEL_STATS.sort_values(["short_scenario", "model"]).reset_index(drop=True)

# 3) Per-scenario pairwise model comparison.
pairwise_rows = []
for scenario_id, g in valid_local.groupby("scenario_id", dropna=False):
    if scenario_id not in SHORT_SCENARIO_MAP:
        continue
    for i, a in enumerate(MODEL_SPECS):
        for b in MODEL_SPECS[i + 1 :]:
            col_a = f"{a['key']}_all_nodes_complete"
            col_b = f"{b['key']}_all_nodes_complete"
            if col_a not in g.columns or col_b not in g.columns:
                continue
            v = g[[col_a, col_b]].dropna()
            if v.empty:
                continue

            rate_a = 100.0 * v[col_a].astype(bool).mean()
            rate_b = 100.0 * v[col_b].astype(bool).mean()
            diff = rate_a - rate_b

            a_only = int((v[col_a].astype(bool) & ~v[col_b].astype(bool)).sum())
            b_only = int((~v[col_a].astype(bool) & v[col_b].astype(bool)).sum())
            if (a_only + b_only) > 0:
                chi2_ab = ((abs(a_only - b_only) - 1) ** 2) / (a_only + b_only)
                p_ab = float(math.erfc(math.sqrt(chi2_ab / 2.0)))
            else:
                chi2_ab = np.nan
                p_ab = np.nan

            pairwise_rows.append({
                "scenario_id": scenario_id,
                "short_scenario": SHORT_SCENARIO_MAP[scenario_id],
                "model_a": a["label"],
                "model_b": b["label"],
                "episodes_eval": int(len(v)),
                "rate_a_pct": round(rate_a, 1),
                "rate_b_pct": round(rate_b, 1),
                "delta_a_minus_b_pct": round(diff, 1),
                "discordant_a_only": a_only,
                "discordant_b_only": b_only,
                "mcnemar_chi2_cc": round(chi2_ab, 4) if pd.notna(chi2_ab) else np.nan,
                "mcnemar_p_approx": round(p_ab, 6) if pd.notna(p_ab) else np.nan,
            })

SCENARIO_PAIRWISE_COMPARE = pd.DataFrame(pairwise_rows)
if not SCENARIO_PAIRWISE_COMPARE.empty:
    SCENARIO_PAIRWISE_COMPARE["short_scenario"] = pd.Categorical(
        SCENARIO_PAIRWISE_COMPARE["short_scenario"],
        categories=SCENARIO_ORDER,
        ordered=True,
    )
    SCENARIO_PAIRWISE_COMPARE = SCENARIO_PAIRWISE_COMPARE.sort_values(["short_scenario", "model_a", "model_b"]).reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(Markdown("## Per-Scenario Summary Tables (Short Names)"))
display(Markdown("### Scenario-Level Rate Summary (with benchmark comparison)"))
display(
    SCENARIO_RATE_TABLE_SHORT[[
        "short_scenario",
        "model",
        "metric",
        "episodes",
        "successes",
        "rate_pct",
        "ci95_lo_pct",
        "ci95_hi_pct",
        "benchmark_success_rate_pct",
        "delta_vs_benchmark_pct",
        "benchmark_within_ci95",
        "outside_95moe_proxy",
    ]]
)

display(Markdown("### Per-Scenario Model Diagnostics vs Simulator"))
display(
    SCENARIO_MODEL_STATS[[
        "short_scenario",
        "model",
        "episodes_eval",
        "pred_success_rate_pct",
        "sim_success_rate_pct",
        "benchmark_success_rate_pct",
        "tp",
        "fp",
        "tn",
        "fn",
        "precision",
        "recall",
        "specificity",
        "accuracy",
        "f1",
        "mcnemar_chi2_cc",
        "mcnemar_p_approx",
    ]]
)

display(Markdown("### Per-Scenario Pairwise Model Comparison"))
display(
    SCENARIO_PAIRWISE_COMPARE[[
        "short_scenario",
        "model_a",
        "model_b",
        "episodes_eval",
        "rate_a_pct",
        "rate_b_pct",
        "delta_a_minus_b_pct",
        "discordant_a_only",
        "discordant_b_only",
        "mcnemar_chi2_cc",
        "mcnemar_p_approx",
    ]]
)

## Per-Scenario Visualization Panel

This section adds five plots built from the per-scenario summary tables:
1. Grouped bar chart of predicted BT success rate by model and scenario.
2. False-positive vs false-negative imbalance plot.
3. Pairwise discordance plot (`discordant_a_only` vs `discordant_b_only`).
4. Heatmap of pairwise model rate differences.
5. Confusion matrix panels by scenario and model.

In [ ]:
import matplotlib.pyplot as plt

if "SCENARIO_MODEL_STATS" not in globals() or SCENARIO_MODEL_STATS.empty:
    raise ValueError("SCENARIO_MODEL_STATS is missing or empty. Run the per-scenario summary cell first.")
if "SCENARIO_PAIRWISE_COMPARE" not in globals() or SCENARIO_PAIRWISE_COMPARE.empty:
    raise ValueError("SCENARIO_PAIRWISE_COMPARE is missing or empty. Run the per-scenario summary cell first.")

# Global typography tuning for clearer presentation.
TITLE_FS = 18
SUBTITLE_FS = 16
LABEL_FS = 13
TICK_FS = 12
LEGEND_FS = 12
ANNOT_FS = 10

scenario_order = [s for s in SCENARIO_ORDER if s in SCENARIO_MODEL_STATS["short_scenario"].astype(str).tolist()]
model_order = [spec["label"] for spec in MODEL_SPECS]

# 1) Grouped bar: predicted BT success rate by model and scenario.
pred_tbl = (
    SCENARIO_MODEL_STATS[["short_scenario", "model", "pred_success_rate_pct"]]
    .copy()
    .pivot(index="short_scenario", columns="model", values="pred_success_rate_pct")
    .reindex(index=scenario_order, columns=model_order)
    .fillna(np.nan)
)

x = np.arange(len(pred_tbl.index))
n_models = len(pred_tbl.columns)
w = 0.8 / max(n_models, 1)

fig, ax = plt.subplots(figsize=(11.5, 5.0))
for i, m in enumerate(pred_tbl.columns):
    offset = (i - (n_models - 1) / 2.0) * w
    ax.bar(x + offset, pred_tbl[m].values, width=w, label=m)

ax.set_xticks(x)
ax.set_xticklabels(pred_tbl.index, rotation=12, ha="right", fontsize=TICK_FS)
ax.set_ylim(0, 105)
ax.set_ylabel("Predicted BT success rate (%)", fontsize=LABEL_FS)
ax.set_xlabel("Scenario", fontsize=LABEL_FS)
ax.set_title("Predicted BT Success Rate by Model and Scenario", fontsize=TITLE_FS)
ax.legend(loc="best", fontsize=LEGEND_FS)
plt.tight_layout()
plt.show()

# 2) FP vs FN imbalance plot (FP - FN).
imbalance_tbl = (
    SCENARIO_MODEL_STATS[["short_scenario", "model", "fp", "fn"]].copy()
    .assign(fp_minus_fn=lambda d: d["fp"] - d["fn"])
    .pivot(index="short_scenario", columns="model", values="fp_minus_fn")
    .reindex(index=scenario_order, columns=model_order)
)

x = np.arange(len(imbalance_tbl.index))
fig, ax = plt.subplots(figsize=(11.5, 5.0))
for i, m in enumerate(imbalance_tbl.columns):
    offset = (i - (n_models - 1) / 2.0) * w
    ax.bar(x + offset, imbalance_tbl[m].values, width=w, label=m)

ax.axhline(0.0, color="gray", linestyle="--", linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(imbalance_tbl.index, rotation=12, ha="right", fontsize=TICK_FS)
ax.set_ylabel("FP - FN (episode count)", fontsize=LABEL_FS)
ax.set_xlabel("Scenario", fontsize=LABEL_FS)
ax.set_title("False Positive vs False Negative Imbalance by Model and Scenario", fontsize=TITLE_FS)
ax.legend(loc="best", fontsize=LEGEND_FS)
plt.tight_layout()
plt.show()

# 3) Pairwise discordance plot (discordant_a_only vs discordant_b_only).
pair_plot = SCENARIO_PAIRWISE_COMPARE.copy()
pair_plot["pair_label"] = pair_plot["model_a"] + " vs " + pair_plot["model_b"]

n_scen = len(scenario_order)
fig, axes = plt.subplots(1, n_scen, figsize=(6.0 * n_scen, 5.0), sharey=True)
if n_scen == 1:
    axes = [axes]

for ax, sc in zip(axes, scenario_order):
    g = pair_plot[pair_plot["short_scenario"].astype(str) == sc].copy()
    g = g.sort_values(["model_a", "model_b"]).reset_index(drop=True)
    x = np.arange(len(g))
    w2 = 0.35
    ax.bar(x - w2 / 2, g["discordant_a_only"], width=w2, label="discordant_a_only")
    ax.bar(x + w2 / 2, g["discordant_b_only"], width=w2, label="discordant_b_only")
    ax.set_xticks(x)
    ax.set_xticklabels(g["pair_label"], rotation=20, ha="right", fontsize=TICK_FS)
    ax.set_title(sc, fontsize=SUBTITLE_FS)
    ax.set_xlabel("Model pair", fontsize=LABEL_FS)
    if ax is axes[0]:
        ax.set_ylabel("Episode count", fontsize=LABEL_FS)

handles, labels = axes[0].get_legend_handles_labels()
# Place legend outside/right to avoid title overlap.
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=LEGEND_FS)
fig.suptitle("Pairwise Discordance by Scenario", y=1.03, fontsize=TITLE_FS)
plt.tight_layout(rect=[0, 0, 0.90, 0.96])
plt.show()

# 4) Heatmap of pairwise model rate differences (row model minus column model).
rate_map = (
    SCENARIO_MODEL_STATS[["short_scenario", "model", "pred_success_rate_pct"]]
    .drop_duplicates()
    .pivot(index="short_scenario", columns="model", values="pred_success_rate_pct")
    .reindex(index=scenario_order, columns=model_order)
)

all_diffs = []
for sc in scenario_order:
    rates = rate_map.loc[sc].values.astype(float)
    d = rates.reshape(-1, 1) - rates.reshape(1, -1)
    all_diffs.append(d)
vmax = np.nanmax(np.abs(np.concatenate(all_diffs))) if len(all_diffs) else 1.0
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0

fig, axes = plt.subplots(1, n_scen, figsize=(5.2 * n_scen + 1.2, 4.8), sharey=True)
if n_scen == 1:
    axes = [axes]

for ax, sc in zip(axes, scenario_order):
    rates = rate_map.loc[sc].values.astype(float)
    delta = rates.reshape(-1, 1) - rates.reshape(1, -1)
    im = ax.imshow(delta, cmap="coolwarm", vmin=-vmax, vmax=vmax)
    ax.set_xticks(np.arange(len(model_order)))
    ax.set_yticks(np.arange(len(model_order)))
    ax.set_xticklabels(model_order, rotation=20, ha="right", fontsize=TICK_FS)
    ax.set_yticklabels(model_order, fontsize=TICK_FS)
    ax.set_title(sc, fontsize=SUBTITLE_FS)
    for r in range(delta.shape[0]):
        for c in range(delta.shape[1]):
            val = delta[r, c]
            txt = f"{val:.1f}" if np.isfinite(val) else "nan"
            ax.text(c, r, txt, ha="center", va="center", fontsize=ANNOT_FS)

# Reserve right margin and draw colorbar in dedicated axis, fully outside heatmaps.
fig.subplots_adjust(right=0.88, top=0.86, wspace=0.25)
cax = fig.add_axes([0.90, 0.20, 0.015, 0.62])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("Rate difference (pct points): row - column", fontsize=LABEL_FS)
fig.suptitle("Pairwise Rate-Difference Heatmaps", y=0.96, fontsize=TITLE_FS)
plt.show()

# 5) Confusion matrix panels (rows=actual, cols=pred).
fig, axes = plt.subplots(len(scenario_order), len(model_order), figsize=(4.8 * len(model_order), 4.2 * len(scenario_order)))
if len(scenario_order) == 1 and len(model_order) == 1:
    axes = np.array([[axes]])
elif len(scenario_order) == 1:
    axes = np.array([axes])
elif len(model_order) == 1:
    axes = np.array([[a] for a in axes])

for i, sc in enumerate(scenario_order):
    for j, m in enumerate(model_order):
        ax = axes[i, j]
        row = SCENARIO_MODEL_STATS[
            (SCENARIO_MODEL_STATS["short_scenario"].astype(str) == sc)
            & (SCENARIO_MODEL_STATS["model"].astype(str) == m)
        ]
        if row.empty:
            mat = np.array([[np.nan, np.nan], [np.nan, np.nan]])
        else:
            rr = row.iloc[0]
            # [[TP, FN], [FP, TN]] where rows are actual success/failure and cols are predicted success/failure
            mat = np.array([[rr["tp"], rr["fn"]], [rr["fp"], rr["tn"]]], dtype=float)

        vmax_cm = np.nanmax(mat) if np.isfinite(mat).any() else 1.0
        if not np.isfinite(vmax_cm) or vmax_cm == 0:
            vmax_cm = 1.0
        _ = ax.imshow(mat, cmap="Blues", vmin=0, vmax=vmax_cm)

        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred: Success", "Pred: Failure"], rotation=15, ha="right", fontsize=TICK_FS)
        ax.set_yticklabels(["Actual: Success", "Actual: Failure"], fontsize=TICK_FS)
        ax.set_title(f"{sc} | {m}", fontsize=SUBTITLE_FS)

        for r in range(2):
            for c in range(2):
                v = mat[r, c]
                txt = "nan" if not np.isfinite(v) else str(int(v))
                color = "white" if np.isfinite(v) and v > vmax_cm / 2 else "black"
                ax.text(c, r, txt, ha="center", va="center", color=color, fontsize=ANNOT_FS, fontweight="bold")

fig.suptitle("Confusion Matrix Panels by Scenario and Model", y=1.01, fontsize=TITLE_FS)
plt.tight_layout()
plt.show()

## Visualization interpretation

### Chart 1: Predicted BT success rate by model and scenario

The predicted BT success-rate chart shows clear differences in model behavior. The original qwen3 model was consistently conservative, predicting completion in only 3%–15% of episodes. Qwen3.5 was much more completion-positive, predicting completion in 83%–100% of episodes. The qwen3.5 mod1 prompt reduced this tendency, producing intermediate completion rates, although the amount of correction varied by scenario.

### Chart 2: False-positive vs. false-negative imbalance

This chart shows the direction of each model’s disagreement with the simulator. Negative values indicate more false negatives, meaning the model under-predicted completion. Positive values indicate more false positives, meaning the model over-predicted completion. Qwen3 showed a strong false-negative pattern across all scenarios, while qwen3.5 showed a consistent false-positive pattern. The mod1 prompt reduced qwen3.5’s over-prediction, but shifted toward under-prediction in Cuttingboard-to-basket and Plate-to-plate.

### Chart 3: Pairwise discordance by scenario

The pairwise discordance chart shows which model predicted completion when the other did not. Across all scenarios, qwen3.5 predicted completion far more often than qwen3, confirming that qwen3.5 was substantially more permissive. Comparisons between qwen3.5 and qwen3.5 mod1 show that mod1 reduced qwen3.5’s completion-positive behavior, with many episodes marked complete only by qwen3.5 and few or none marked complete only by mod1.

### Chart 4: Pairwise rate-difference heatmaps

The heatmaps summarize the size and direction of completion-rate differences between models. Qwen3 had much lower completion rates than qwen3.5 across all scenarios, with differences of roughly 80 percentage points. Qwen3.5 also predicted completion more often than qwen3.5 mod1 in every scenario. Overall, the heatmaps reinforce the main pattern: qwen3 was the most conservative model, qwen3.5 was the most permissive model, and qwen3.5 mod1 generally fell between them.

### Chart 5: Confusion matrix panels

The confusion matrices show how each model aligned with simulator outcomes. Qwen3 produced many false negatives, indicating that it often missed simulator-success episodes. Qwen3.5 produced high true-positive counts but also many false positives, indicating high recall but low specificity. Qwen3.5 mod1 reduced the extreme over-prediction seen in qwen3.5, but its performance was scenario-dependent. It improved balance for Can-to-drawer-close, became too conservative for Cuttingboard-to-basket, and partially corrected qwen3.5’s over-prediction in Plate-to-plate.

## Overall takeaway

Across the visualizations, qwen3 was overly conservative, qwen3.5 was overly permissive, and qwen3.5 mod1 partially corrected qwen3.5’s completion-positive bias. However, the correction was not uniform across scenarios, suggesting that prompt tuning improved model behavior but may still require scenario-specific calibration, or there is additional interpretation in actions that can be explored.


## Reviewer vs Model Node-Level Audit (Reviewed Episodes Only)

This section treats reviewer annotations as ground truth and compares each model against reviewer labels on reviewed episodes only.

Outputs include:
- Coverage and sample-size summary for reviewed episodes/nodes.
- Node-level completion and failure rate comparison (reviewer vs model).
- Node-level confusion metrics with reviewer as truth for completion and failure labels.
- Timing-difference tables for completion and failure timestamps (model time minus reviewer time).
- Episode-level comparisons with reviewer-derived success as truth, including simulator (`success`) as a subset reference.

In [ ]:
if "episode_df_2_21" not in globals() or episode_df_2_21.empty:
    raise ValueError("episode_df_2_21 is missing or empty. Run Cell 1 first.")

audit_df = episode_df_2_21.copy()

def _to_bool(v: object) -> bool:
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if pd.isna(v):
        return False
    s = str(v).strip().lower()
    return s in {"true", "1", "yes", "y", "t"}

def _is_filled(v: object) -> bool:
    if pd.isna(v):
        return False
    return str(v).strip() != ""

def _to_seconds_mmss(v: object) -> float:
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "":
        return np.nan
    if ":" in s:
        parts = s.split(":")
        try:
            if len(parts) == 2:
                m = float(parts[0])
                sec = float(parts[1])
                return 60.0 * m + sec
            if len(parts) == 3:
                h = float(parts[0])
                m = float(parts[1])
                sec = float(parts[2])
                return 3600.0 * h + 60.0 * m + sec
        except Exception:
            return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def _conf_metrics(truth: pd.Series, pred: pd.Series) -> dict[str, float]:
    t = truth.astype(bool)
    p = pred.astype(bool)
    tp = int((t & p).sum())
    fp = int((~t & p).sum())
    tn = int((~t & ~p).sum())
    fn = int((t & ~p).sum())
    n = int(len(t))
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    accuracy = (tp + tn) / n if n else np.nan
    f1 = (2 * precision * recall / (precision + recall)) if (pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0) else np.nan
    return {
        "n": n,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1,
    }

# Reviewed-episode filter: prefer explicit flag; fallback to reviewer timestamp evidence.
if "human_reviewed" in audit_df.columns:
    reviewed_mask = audit_df["human_reviewed"].map(_to_bool)
else:
    reviewed_mask = audit_df["node_1_reviewer_completion_time"].map(_is_filled) if "node_1_reviewer_completion_time" in audit_df.columns else pd.Series(False, index=audit_df.index)

reviewed_eps = audit_df[reviewed_mask].copy()
reviewed_eps["sim_success_bool"] = reviewed_eps["success"].map(_to_bool)

MODEL_REVIEW_SPECS = [
    {"key": "qwen3", "label": "qwen3", "comp_suffix": "model_completion_time", "fail_suffix": "model_failure_time"},
    {"key": "qwen35", "label": "qwen3.5", "comp_suffix": "qwen3_5_model_completion_time", "fail_suffix": "qwen3_5_model_failure_time"},
    {"key": "qwen35_mod1", "label": "qwen3.5 mod1", "comp_suffix": "qwen3_5_mod1_completion_time", "fail_suffix": "qwen3_5_mod1_failure_time"},
]

# Build node-level long dataframe over required action nodes only.
node_rows: list[dict[str, object]] = []
for _, row in reviewed_eps.iterrows():
    required_nodes = max(0, int(pd.to_numeric(row.get("bt_action_nodes", 0), errors="coerce") if pd.notna(row.get("bt_action_nodes", np.nan)) else 0))
    for i in range(1, required_nodes + 1):
        rev_comp_raw = row.get(f"node_{i}_reviewer_completion_time", np.nan)
        rev_fail_raw = row.get(f"node_{i}_reviewer_failure_time", np.nan)
        rev_comp = _is_filled(rev_comp_raw)
        rev_fail = _is_filled(rev_fail_raw)

        rec = {
            "episode_dir": row.get("episode_dir", ""),
            "scenario_id": row.get("scenario_id", ""),
            "episode_index": row.get("episode_index", np.nan),
            "node_index": i,
            "node_id": row.get(f"node_{i}_node_id", ""),
            "sim_success_bool": _to_bool(row.get("success", False)),
            "reviewer_complete": rev_comp,
            "reviewer_failed": rev_fail,
            "reviewer_completion_s": _to_seconds_mmss(rev_comp_raw),
            "reviewer_failure_s": _to_seconds_mmss(rev_fail_raw),
        }

        for spec in MODEL_REVIEW_SPECS:
            m_comp_raw = row.get(f"node_{i}_{spec['comp_suffix']}", np.nan)
            m_fail_raw = row.get(f"node_{i}_{spec['fail_suffix']}", np.nan)
            rec[f"{spec['key']}_complete"] = _is_filled(m_comp_raw)
            rec[f"{spec['key']}_failed"] = _is_filled(m_fail_raw)
            rec[f"{spec['key']}_completion_s"] = _to_seconds_mmss(m_comp_raw)
            rec[f"{spec['key']}_failure_s"] = _to_seconds_mmss(m_fail_raw)

        node_rows.append(rec)

NODE_REVIEW_DF = pd.DataFrame(node_rows)

# Coverage summary (target: 60 reviewed episodes).
REVIEW_COVERAGE = pd.DataFrame([
    {
        "reviewed_episode_count": int(reviewed_eps["episode_dir"].nunique()) if not reviewed_eps.empty else 0,
        "reviewed_row_count": int(len(reviewed_eps)),
        "reviewed_node_count": int(len(NODE_REVIEW_DF)),
        "reviewed_scenario_count": int(reviewed_eps["scenario_id"].nunique()) if not reviewed_eps.empty else 0,
    }
])

# Node-level rate summary (reviewer truth vs model rates).
rate_rows = []
for spec in MODEL_REVIEW_SPECS:
    rate_rows.append({
        "model": spec["label"],
        "nodes_eval": int(len(NODE_REVIEW_DF)),
        "reviewer_completion_rate_pct": round(100.0 * NODE_REVIEW_DF["reviewer_complete"].mean(), 1) if not NODE_REVIEW_DF.empty else np.nan,
        "model_completion_rate_pct": round(100.0 * NODE_REVIEW_DF[f"{spec['key']}_complete"].mean(), 1) if not NODE_REVIEW_DF.empty else np.nan,
        "reviewer_failure_rate_pct": round(100.0 * NODE_REVIEW_DF["reviewer_failed"].mean(), 1) if not NODE_REVIEW_DF.empty else np.nan,
        "model_failure_rate_pct": round(100.0 * NODE_REVIEW_DF[f"{spec['key']}_failed"].mean(), 1) if not NODE_REVIEW_DF.empty else np.nan,
    })
NODE_RATE_SUMMARY = pd.DataFrame(rate_rows)

# Completion/failure confusion tables with reviewer as truth.
completion_rows = []
failure_rows = []
for spec in MODEL_REVIEW_SPECS:
    if NODE_REVIEW_DF.empty:
        continue

    cm = _conf_metrics(NODE_REVIEW_DF["reviewer_complete"], NODE_REVIEW_DF[f"{spec['key']}_complete"])
    completion_rows.append({
        "model": spec["label"],
        **cm,
    })

    fm = _conf_metrics(NODE_REVIEW_DF["reviewer_failed"], NODE_REVIEW_DF[f"{spec['key']}_failed"])
    failure_rows.append({
        "model": spec["label"],
        **fm,
    })

NODE_COMPLETION_CONFUSION = pd.DataFrame(completion_rows)
NODE_FAILURE_CONFUSION = pd.DataFrame(failure_rows)

# Timing differences (model minus reviewer, seconds).
timing_completion_rows = []
timing_failure_rows = []
for spec in MODEL_REVIEW_SPECS:
    if NODE_REVIEW_DF.empty:
        continue

    comp_mask = NODE_REVIEW_DF["reviewer_complete"] & NODE_REVIEW_DF[f"{spec['key']}_complete"]
    comp_pairs = NODE_REVIEW_DF.loc[comp_mask, ["reviewer_completion_s", f"{spec['key']}_completion_s"]].dropna()
    if not comp_pairs.empty:
        delta = comp_pairs[f"{spec['key']}_completion_s"] - comp_pairs["reviewer_completion_s"]
        timing_completion_rows.append({
            "model": spec["label"],
            "pairs_n": int(len(comp_pairs)),
            "mean_delta_s": round(float(delta.mean()), 3),
            "median_delta_s": round(float(delta.median()), 3),
            "mae_s": round(float(delta.abs().mean()), 3),
            "rmse_s": round(float(np.sqrt((delta ** 2).mean())), 3),
            "pct_within_2s": round(float(100.0 * (delta.abs() <= 2.0).mean()), 1),
        })
    else:
        timing_completion_rows.append({
            "model": spec["label"],
            "pairs_n": 0,
            "mean_delta_s": np.nan,
            "median_delta_s": np.nan,
            "mae_s": np.nan,
            "rmse_s": np.nan,
            "pct_within_2s": np.nan,
        })

    fail_mask = NODE_REVIEW_DF["reviewer_failed"] & NODE_REVIEW_DF[f"{spec['key']}_failed"]
    fail_pairs = NODE_REVIEW_DF.loc[fail_mask, ["reviewer_failure_s", f"{spec['key']}_failure_s"]].dropna()
    if not fail_pairs.empty:
        delta_f = fail_pairs[f"{spec['key']}_failure_s"] - fail_pairs["reviewer_failure_s"]
        timing_failure_rows.append({
            "model": spec["label"],
            "pairs_n": int(len(fail_pairs)),
            "mean_delta_s": round(float(delta_f.mean()), 3),
            "median_delta_s": round(float(delta_f.median()), 3),
            "mae_s": round(float(delta_f.abs().mean()), 3),
            "rmse_s": round(float(np.sqrt((delta_f ** 2).mean())), 3),
            "pct_within_2s": round(float(100.0 * (delta_f.abs() <= 2.0).mean()), 1),
        })
    else:
        timing_failure_rows.append({
            "model": spec["label"],
            "pairs_n": 0,
            "mean_delta_s": np.nan,
            "median_delta_s": np.nan,
            "mae_s": np.nan,
            "rmse_s": np.nan,
            "pct_within_2s": np.nan,
        })

TIMING_COMPLETION_SUMMARY = pd.DataFrame(timing_completion_rows)
TIMING_FAILURE_SUMMARY = pd.DataFrame(timing_failure_rows)

# Episode-level reviewer-truth comparisons.
def _episode_truth_from_row(row: pd.Series, comp_suffix: str) -> bool:
    n = max(0, int(pd.to_numeric(row.get("bt_action_nodes", 0), errors="coerce") if pd.notna(row.get("bt_action_nodes", np.nan)) else 0))
    if n == 0:
        return False
    for i in range(1, n + 1):
        if not _is_filled(row.get(f"node_{i}_{comp_suffix}", np.nan)):
            return False
    return True

ep = reviewed_eps.copy()
ep["reviewer_episode_success"] = ep.apply(lambda r: _episode_truth_from_row(r, "reviewer_completion_time"), axis=1)
for spec in MODEL_REVIEW_SPECS:
    ep[f"{spec['key']}_episode_success"] = ep.apply(
        lambda r, s=spec["comp_suffix"]: _episode_truth_from_row(r, s),
        axis=1,
    )
ep["sim_episode_success"] = ep["success"].map(_to_bool)

# Primary: model vs reviewer truth.
episode_rows = []
for spec in MODEL_REVIEW_SPECS:
    m = _conf_metrics(ep["reviewer_episode_success"], ep[f"{spec['key']}_episode_success"])
    episode_rows.append({
        "predictor": spec["label"],
        "reviewer_success_rate_pct": round(100.0 * ep["reviewer_episode_success"].mean(), 1) if not ep.empty else np.nan,
        "predictor_success_rate_pct": round(100.0 * ep[f"{spec['key']}_episode_success"].mean(), 1) if not ep.empty else np.nan,
        **m,
    })

EPISODE_VS_REVIEWER_SUMMARY = pd.DataFrame(episode_rows)

# Subset reference: simulator vs reviewer truth.
sim_vs_review = _conf_metrics(ep["reviewer_episode_success"], ep["sim_episode_success"]) if not ep.empty else {
    "n": 0, "tp": 0, "fp": 0, "tn": 0, "fn": 0, "precision": np.nan, "recall": np.nan, "specificity": np.nan, "accuracy": np.nan, "f1": np.nan
}
SYSTEM_VS_REVIEWER_SUMMARY = pd.DataFrame([
    {
        "predictor": "simulator_success",
        "reviewer_success_rate_pct": round(100.0 * ep["reviewer_episode_success"].mean(), 1) if not ep.empty else np.nan,
        "predictor_success_rate_pct": round(100.0 * ep["sim_episode_success"].mean(), 1) if not ep.empty else np.nan,
        **sim_vs_review,
    }
])

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(Markdown("## Reviewer-Truth Audit Tables (Reviewed Episodes Only)"))
display(Markdown("### Coverage"))
display(REVIEW_COVERAGE)

display(Markdown("### Node-Level Completion/Failure Rates: Reviewer vs Models"))
display(NODE_RATE_SUMMARY)

display(Markdown("### Node-Level Completion Confusion (Reviewer = Truth)"))
display(NODE_COMPLETION_CONFUSION)

display(Markdown("### Node-Level Failure Confusion (Reviewer = Truth)"))
display(NODE_FAILURE_CONFUSION)

display(Markdown("### Timing Deltas for Completion Events (Model - Reviewer, seconds)"))
display(TIMING_COMPLETION_SUMMARY)

display(Markdown("### Timing Deltas for Failure Events (Model - Reviewer, seconds)"))
display(TIMING_FAILURE_SUMMARY)

display(Markdown("### Episode-Level: Model vs Reviewer Truth (all reviewed episodes)"))
display(EPISODE_VS_REVIEWER_SUMMARY)

display(Markdown("### Episode-Level Subset: Simulator vs Reviewer Truth"))
display(SYSTEM_VS_REVIEWER_SUMMARY)

if not REVIEW_COVERAGE.empty:
    n_reviewed = int(REVIEW_COVERAGE.loc[0, "reviewed_episode_count"])
    display(Markdown(
        f"Reviewed episode count used in this section: **{n_reviewed}**."
    ))

## Reviewer vs Model Node-Level Audit Visualizations

These plots summarize how each model aligns with reviewer labels at the node level.

Figures in this cell:
1. Grouped bar chart of node-level metric profile by model (precision, recall, specificity, accuracy, F1).
2. Reviewer vs model completion and failure rates.
3. Node-level confusion matrix panels by model (completion and failure).

In [ ]:
import matplotlib.pyplot as plt

if "NODE_RATE_SUMMARY" not in globals() or NODE_RATE_SUMMARY.empty:
    raise ValueError("NODE_RATE_SUMMARY is missing or empty. Run the reviewer audit tables cell first.")
if "NODE_COMPLETION_CONFUSION" not in globals() or NODE_COMPLETION_CONFUSION.empty:
    raise ValueError("NODE_COMPLETION_CONFUSION is missing or empty. Run the reviewer audit tables cell first.")
if "NODE_FAILURE_CONFUSION" not in globals() or NODE_FAILURE_CONFUSION.empty:
    raise ValueError("NODE_FAILURE_CONFUSION is missing or empty. Run the reviewer audit tables cell first.")

model_order = [spec["label"] for spec in MODEL_REVIEW_SPECS] if "MODEL_REVIEW_SPECS" in globals() else NODE_RATE_SUMMARY["model"].tolist()

# 1) Grouped bar chart: node-level metric profile by model.
metric_cols = ["precision", "recall", "specificity", "accuracy", "f1"]
completion_metric_tbl = (
    NODE_COMPLETION_CONFUSION[["model"] + metric_cols]
    .copy()
    .set_index("model")
    .reindex(model_order)
)
failure_metric_tbl = (
    NODE_FAILURE_CONFUSION[["model"] + metric_cols]
    .copy()
    .set_index("model")
    .reindex(model_order)
)

# Keep a mask of undefined metrics (e.g., precision when no positive predictions),
# then plot zeros for visibility and annotate those bars as n/a.
completion_nan_mask = completion_metric_tbl.isna()
failure_nan_mask = failure_metric_tbl.isna()
completion_plot_tbl = completion_metric_tbl.fillna(0.0)
failure_plot_tbl = failure_metric_tbl.fillna(0.0)

x = np.arange(len(metric_cols))
bar_w = 0.8 / max(len(model_order), 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
for i, model_label in enumerate(model_order):
    offset = (i - (len(model_order) - 1) / 2.0) * bar_w
    bars_c = axes[0].bar(
        x + offset,
        completion_plot_tbl.loc[model_label].values.astype(float),
        width=bar_w,
        label=model_label,
    )
    bars_f = axes[1].bar(
        x + offset,
        failure_plot_tbl.loc[model_label].values.astype(float),
        width=bar_w,
        label=model_label,
    )

    # Mark undefined bars as n/a on top of zero-height bars.
    for k, metric in enumerate(metric_cols):
        if completion_nan_mask.loc[model_label, metric]:
            axes[0].text(
                x[k] + offset,
                0.03,
                "n/a",
                ha="center",
                va="bottom",
                rotation=90,
                fontsize=8,
                color="dimgray",
            )
        if failure_nan_mask.loc[model_label, metric]:
            axes[1].text(
                x[k] + offset,
                0.03,
                "n/a",
                ha="center",
                va="bottom",
                rotation=90,
                fontsize=8,
                color="dimgray",
            )

for ax, title in zip(axes, ["Completion Metrics", "Failure Metrics"]):
    ax.set_xticks(x)
    ax.set_xticklabels(metric_cols, rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle("Node-Level Metric Profile by Model (Reviewer = Truth)", fontsize=15)
plt.tight_layout(rect=[0, 0, 0.93, 0.95])
plt.show()

# 2) Reviewer vs model completion/failure rates.
rate_tbl = NODE_RATE_SUMMARY.copy().set_index("model").reindex(model_order)
reviewer_completion_rate = float(rate_tbl["reviewer_completion_rate_pct"].mean())
reviewer_failure_rate = float(rate_tbl["reviewer_failure_rate_pct"].mean())

x = np.arange(len(model_order))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Completion rates
axes[0].bar(x - w / 2, [reviewer_completion_rate] * len(model_order), width=w, label="reviewer", color="#4E79A7")
axes[0].bar(x + w / 2, rate_tbl["model_completion_rate_pct"].values.astype(float), width=w, label="model", color="#F28E2B")
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_order, rotation=15, ha="right")
axes[0].set_ylabel("Rate (%)")
axes[0].set_ylim(0, 105)
axes[0].set_title("Completion Rate: Reviewer vs Model")
axes[0].grid(axis="y", linestyle="--", alpha=0.3)

# Failure rates
axes[1].bar(x - w / 2, [reviewer_failure_rate] * len(model_order), width=w, label="reviewer", color="#4E79A7")
axes[1].bar(x + w / 2, rate_tbl["model_failure_rate_pct"].values.astype(float), width=w, label="model", color="#E15759")
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_order, rotation=15, ha="right")
axes[1].set_ylabel("Rate (%)")
axes[1].set_ylim(0, 105)
axes[1].set_title("Failure Rate: Reviewer vs Model")
axes[1].grid(axis="y", linestyle="--", alpha=0.3)

# Place legend below title and reserve top margin to avoid overlap.
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.97), ncol=2, frameon=False)
fig.suptitle("Reviewer vs Model Node-Level Rates", fontsize=19, y=1.03)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.show()

# 3) Node-level confusion matrix panels by model.
comp_tbl = NODE_COMPLETION_CONFUSION.set_index("model").reindex(model_order)
fail_tbl = NODE_FAILURE_CONFUSION.set_index("model").reindex(model_order)

fig, axes = plt.subplots(len(model_order), 2, figsize=(10.5, 4.0 * len(model_order)))
if len(model_order) == 1:
    axes = np.array([axes])

for i, model_label in enumerate(model_order):
    comp = comp_tbl.loc[model_label]
    fail = fail_tbl.loc[model_label]

    mats = [
        np.array([[comp["tp"], comp["fn"]], [comp["fp"], comp["tn"]]], dtype=float),
        np.array([[fail["tp"], fail["fn"]], [fail["fp"], fail["tn"]]], dtype=float),
    ]
    titles = [f"{model_label} | Completion", f"{model_label} | Failure"]

    for j in range(2):
        ax = axes[i, j]
        mat = mats[j]
        vmax = np.nanmax(mat) if np.isfinite(mat).any() else 1.0
        if not np.isfinite(vmax) or vmax <= 0:
            vmax = 1.0

        ax.imshow(mat, cmap="Blues", vmin=0, vmax=vmax)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["Pred: Yes", "Pred: No"], rotation=15, ha="right")
        ax.set_yticklabels(["True: Yes", "True: No"])
        ax.set_title(titles[j])

        for r in range(2):
            for c in range(2):
                v = mat[r, c]
                txt = "nan" if not np.isfinite(v) else str(int(v))
                color = "white" if np.isfinite(v) and v > (vmax / 2.0) else "black"
                ax.text(c, r, txt, ha="center", va="center", color=color, fontsize=11, fontweight="bold")

fig.suptitle("Node-Level Confusion Matrix Panels by Model (Reviewer = Truth)", fontsize=15, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()## Reviewer Breakdown A: Agreement and Scenario Diagnostics

### Reviewed Episodes Only

The reviewer audit covered 60 reviewed episodes, 145 reviewed BT nodes, and three task scenarios. At the node level, the reviewer marked 73.1% of nodes as complete and 10.3% as failure states. The models showed clear differences in how they used these labels. Qwen3 was conservative, marking only 27.6% of nodes complete, while qwen3.5 was highly completion-positive, marking 91.0% complete. The qwen3.5 mod1 prompt reduced this tendency, marking 60.7% of nodes complete.

The node-level metric profile shows that qwen3 had very high precision and specificity for completion, but low recall, meaning it rarely over-called completion but missed many reviewer-complete nodes. Qwen3.5 showed the opposite pattern, with very high recall but low specificity, meaning it captured most reviewer-complete nodes but also produced more false completions. Qwen3.5 mod1 provided a more balanced profile, reducing qwen3.5’s false-completion tendency while maintaining stronger recall than qwen3.

The failure-rate comparison is also important. Qwen3 predicted failure states, although with some false failure calls. In contrast, qwen3.5 and qwen3.5 mod1 did not predict any failure states in the reviewed node set. This suggests that while qwen3.5-based prompts improved completion detection, they did not yet provide useful explicit failure detection.

## Reviewer Breakdown A: Agreement and Scenario Diagnostics

This section focuses on reviewer-as-truth agreement metrics at the node level, then breaks diagnostics down by scenario and model.

Tables in this section:
- Node-level agreement summary (completion and failure events)
- Scenario-by-model completion diagnostics
- Scenario-by-model failure diagnostics

In [ ]:
if "NODE_REVIEW_DF" not in globals() or NODE_REVIEW_DF.empty:
    raise ValueError("NODE_REVIEW_DF is missing or empty. Run the reviewer-truth audit cell first.")
if "MODEL_REVIEW_SPECS" not in globals() or not MODEL_REVIEW_SPECS:
    raise ValueError("MODEL_REVIEW_SPECS is missing. Run the reviewer-truth audit cell first.")


def _conf_metrics_local(truth: pd.Series, pred: pd.Series) -> dict[str, float]:
    t = truth.astype(bool)
    p = pred.astype(bool)
    tp = int((t & p).sum())
    fp = int((~t & p).sum())
    tn = int((~t & ~p).sum())
    fn = int((t & ~p).sum())
    n = int(len(t))
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    accuracy = (tp + tn) / n if n else np.nan
    f1 = (2 * precision * recall / (precision + recall)) if (pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0) else np.nan
    return {
        "n": n,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1,
    }


def _short_scenario_name_local(s: object) -> str:
    sc = str(s)
    if "SHORT_SCENARIO_MAP" in globals() and isinstance(SHORT_SCENARIO_MAP, dict) and sc in SHORT_SCENARIO_MAP:
        return str(SHORT_SCENARIO_MAP[sc])
    txt = sc.split("/")[-1]
    txt = txt.replace("_GR1ArmsAndWaistFourierHands_Env", "")
    txt = txt.replace("SplitA", "")
    txt = txt.replace("Posttrain", "")
    txt = txt.replace("PnP", "")
    txt = txt.replace("_", " ").strip()
    return txt if txt else sc


agreement_rows = []
for spec in MODEL_REVIEW_SPECS:
    model_key = spec["key"]
    for event_name, reviewer_col, model_col in [
        ("completion", "reviewer_complete", f"{model_key}_complete"),
        ("failure", "reviewer_failed", f"{model_key}_failed"),
    ]:
        truth = NODE_REVIEW_DF[reviewer_col].astype(bool)
        pred = NODE_REVIEW_DF[model_col].astype(bool)
        n = int(len(NODE_REVIEW_DF))
        both_true = int((truth & pred).sum())
        both_false = int((~truth & ~pred).sum())
        reviewer_only = int((truth & ~pred).sum())
        model_only = int((~truth & pred).sum())
        agreement = (both_true + both_false) / n if n else np.nan
        disagreement = (reviewer_only + model_only) / n if n else np.nan

        agreement_rows.append({
            "model": spec["label"],
            "event_type": event_name,
            "n": n,
            "agreement_rate_pct": round(100.0 * agreement, 1) if n else np.nan,
            "disagreement_rate_pct": round(100.0 * disagreement, 1) if n else np.nan,
            "both_true": both_true,
            "both_false": both_false,
            "reviewer_only": reviewer_only,
            "model_only": model_only,
        })

NODE_AGREEMENT_SUMMARY = pd.DataFrame(agreement_rows).sort_values(["event_type", "model"]).reset_index(drop=True)


scenario_completion_rows = []
scenario_failure_rows = []
for scenario_id, g in NODE_REVIEW_DF.groupby("scenario_id", dropna=False):
    short_sc = _short_scenario_name_local(scenario_id)
    for spec in MODEL_REVIEW_SPECS:
        model_key = spec["key"]

        cm = _conf_metrics_local(g["reviewer_complete"], g[f"{model_key}_complete"])
        scenario_completion_rows.append({
            "scenario_id": scenario_id,
            "short_scenario": short_sc,
            "model": spec["label"],
            "reviewer_completion_rate_pct": round(100.0 * g["reviewer_complete"].mean(), 1),
            "model_completion_rate_pct": round(100.0 * g[f"{model_key}_complete"].mean(), 1),
            **cm,
        })

        fm = _conf_metrics_local(g["reviewer_failed"], g[f"{model_key}_failed"])
        scenario_failure_rows.append({
            "scenario_id": scenario_id,
            "short_scenario": short_sc,
            "model": spec["label"],
            "reviewer_failure_rate_pct": round(100.0 * g["reviewer_failed"].mean(), 1),
            "model_failure_rate_pct": round(100.0 * g[f"{model_key}_failed"].mean(), 1),
            **fm,
        })

SCENARIO_MODEL_COMPLETION_DIAGNOSTICS = pd.DataFrame(scenario_completion_rows).sort_values(["short_scenario", "model"]).reset_index(drop=True)
SCENARIO_MODEL_FAILURE_DIAGNOSTICS = pd.DataFrame(scenario_failure_rows).sort_values(["short_scenario", "model"]).reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(Markdown("### Node-Level Agreement with Reviewer (Reviewer = Truth)"))
display(NODE_AGREEMENT_SUMMARY)

display(Markdown("### Scenario-by-Model Completion Diagnostics"))
display(
    SCENARIO_MODEL_COMPLETION_DIAGNOSTICS[[
        "short_scenario",
        "model",
        "reviewer_completion_rate_pct",
        "model_completion_rate_pct",
        "n",
        "tp",
        "fp",
        "tn",
        "fn",
        "precision",
        "recall",
        "specificity",
        "accuracy",
        "f1",
    ]]
)

display(Markdown("### Scenario-by-Model Failure Diagnostics"))
display(
    SCENARIO_MODEL_FAILURE_DIAGNOSTICS[[
        "short_scenario",
        "model",
        "reviewer_failure_rate_pct",
        "model_failure_rate_pct",
        "n",
        "tp",
        "fp",
        "tn",
        "fn",
        "precision",
        "recall",
        "specificity",
        "accuracy",
        "f1",
    ]]
)

## Reviewer Breakdown A Visualizations

These plots visualize scenario and episode agreement behavior from Reviewer Breakdown A.

Figures in this section:
1. Scenario x model metric heatmaps (completion and failure accuracy).
2. Scenario-level FP/FN imbalance against reviewer (FP - FN).
3. Episode-level reviewer agreement by model.

In [ ]:
import matplotlib.pyplot as plt
import re

if "SCENARIO_MODEL_COMPLETION_DIAGNOSTICS" not in globals() or SCENARIO_MODEL_COMPLETION_DIAGNOSTICS.empty:
    raise ValueError("SCENARIO_MODEL_COMPLETION_DIAGNOSTICS is missing or empty. Run Reviewer Breakdown A table cell first.")
if "SCENARIO_MODEL_FAILURE_DIAGNOSTICS" not in globals() or SCENARIO_MODEL_FAILURE_DIAGNOSTICS.empty:
    raise ValueError("SCENARIO_MODEL_FAILURE_DIAGNOSTICS is missing or empty. Run Reviewer Breakdown A table cell first.")
if "reviewed_eps" not in globals() or reviewed_eps.empty:
    raise ValueError("reviewed_eps is missing or empty. Run the reviewer audit tables cell first.")
if "MODEL_REVIEW_SPECS" not in globals() or not MODEL_REVIEW_SPECS:
    raise ValueError("MODEL_REVIEW_SPECS is missing. Run the reviewer audit tables cell first.")


def _short_scenario_name(s: object) -> str:
    sc = str(s)
    if "SHORT_SCENARIO_MAP" in globals() and isinstance(SHORT_SCENARIO_MAP, dict) and sc in SHORT_SCENARIO_MAP:
        return str(SHORT_SCENARIO_MAP[sc])

    # Fallback: trim common suffixes and de-camelize for readability.
    txt = sc.split("/")[-1]
    txt = txt.replace("_GR1ArmsAndWaistFourierHands_Env", "")
    txt = txt.replace("SplitA", "")
    txt = txt.replace("Posttrain", "")
    txt = txt.replace("PnP", "")
    txt = re.sub(r"([a-z])([A-Z])", r"\1 \2", txt)
    txt = re.sub(r"\s+", " ", txt.replace("_", " ")).strip()
    return txt if txt else sc


scenario_ids = sorted(SCENARIO_MODEL_COMPLETION_DIAGNOSTICS["scenario_id"].astype(str).unique().tolist())
scenario_labels = [_short_scenario_name(s) for s in scenario_ids]
model_order = [spec["label"] for spec in MODEL_REVIEW_SPECS]

# 1) Scenario x model metric heatmaps.
comp_heat = (
    SCENARIO_MODEL_COMPLETION_DIAGNOSTICS
    .pivot(index="scenario_id", columns="model", values="accuracy")
    .reindex(index=scenario_ids, columns=model_order)
)
fail_heat = (
    SCENARIO_MODEL_FAILURE_DIAGNOSTICS
    .pivot(index="scenario_id", columns="model", values="accuracy")
    .reindex(index=scenario_ids, columns=model_order)
)

all_vals = np.concatenate([comp_heat.values.flatten(), fail_heat.values.flatten()])
all_vals = all_vals[np.isfinite(all_vals)]
if all_vals.size > 0:
    vmin = float(all_vals.min())
    vmax = float(all_vals.max())
    if np.isclose(vmin, vmax):
        vmin = max(0.0, vmin - 0.05)
        vmax = min(1.0, vmax + 0.05)
else:
    vmin, vmax = 0.0, 1.0

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8), sharey=True, constrained_layout=True)
for ax, mat_df, title in [
    (axes[0], comp_heat, "Completion Accuracy"),
    (axes[1], fail_heat, "Failure Accuracy"),
]:
    mat = mat_df.values.astype(float)
    im = ax.imshow(mat, cmap="YlGnBu", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(model_order)))
    ax.set_yticks(np.arange(len(scenario_labels)))
    ax.set_xticklabels(model_order, rotation=20, ha="right")
    ax.set_yticklabels(scenario_labels)
    ax.set_title(f"Scenario x Model Heatmap: {title}")

    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            v = mat[r, c]
            txt = "nan" if not np.isfinite(v) else f"{v:.2f}"
            color = "white" if np.isfinite(v) and v >= (vmin + 0.6 * (vmax - vmin)) else "black"
            ax.text(c, r, txt, ha="center", va="center", fontsize=9, color=color)

cbar = fig.colorbar(im, ax=axes, shrink=0.92, pad=0.02)
cbar.set_label("Accuracy (shared scale, higher is better)")
fig.suptitle("Reviewer Breakdown A: Scenario x Model Metric Heatmaps", fontsize=19)
plt.show()

# 2) Scenario-level FP/FN imbalance against reviewer (FP - FN).
comp_imb = SCENARIO_MODEL_COMPLETION_DIAGNOSTICS.copy()
comp_imb["imbalance_fp_minus_fn"] = comp_imb["fp"] - comp_imb["fn"]
fail_imb = SCENARIO_MODEL_FAILURE_DIAGNOSTICS.copy()
fail_imb["imbalance_fp_minus_fn"] = fail_imb["fp"] - fail_imb["fn"]

x = np.arange(len(scenario_ids))
w = 0.8 / max(len(model_order), 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.8), sharey=True)
for i, m in enumerate(model_order):
    offset = (i - (len(model_order) - 1) / 2.0) * w

    g_comp = (
        comp_imb[comp_imb["model"] == m]
        .set_index("scenario_id")
        .reindex(scenario_ids)
    )
    g_fail = (
        fail_imb[fail_imb["model"] == m]
        .set_index("scenario_id")
        .reindex(scenario_ids)
    )

    axes[0].bar(x + offset, g_comp["imbalance_fp_minus_fn"].values.astype(float), width=w, label=m)
    axes[1].bar(x + offset, g_fail["imbalance_fp_minus_fn"].values.astype(float), width=w, label=m)

for ax, title in zip(axes, ["Completion FP/FN Imbalance", "Failure FP/FN Imbalance"]):
    ax.axhline(0, color="gray", linestyle="--", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(scenario_labels, rotation=15, ha="right")
    ax.set_ylabel("FP - FN")
    ax.set_title(title)
    ax.grid(axis="y", linestyle="--", alpha=0.3)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle("Scenario-Level FP/FN Imbalance Against Reviewer", fontsize=17)
plt.tight_layout(rect=[0, 0, 0.93, 0.95])
plt.show()

# 3) Episode-level reviewer agreement by model.
def _is_filled_local(v: object) -> bool:
    if pd.isna(v):
        return False
    return str(v).strip() != ""


def _episode_success_from_suffix(row: pd.Series, comp_suffix: str) -> bool:
    n_nodes = max(0, int(pd.to_numeric(row.get("bt_action_nodes", 0), errors="coerce") if pd.notna(row.get("bt_action_nodes", np.nan)) else 0))
    if n_nodes == 0:
        return False
    for i in range(1, n_nodes + 1):
        if not _is_filled_local(row.get(f"node_{i}_{comp_suffix}", np.nan)):
            return False
    return True


ep_local = reviewed_eps.copy()
ep_local["reviewer_episode_success"] = ep_local.apply(lambda r: _episode_success_from_suffix(r, "reviewer_completion_time"), axis=1)

agreement_rows = []
for spec in MODEL_REVIEW_SPECS:
    model_col = f"{spec['key']}_episode_success"
    ep_local[model_col] = ep_local.apply(lambda r, s=spec["comp_suffix"]: _episode_success_from_suffix(r, s), axis=1)

    agree = ep_local["reviewer_episode_success"].astype(bool) == ep_local[model_col].astype(bool)
    agreement_rows.append({
        "model": spec["label"],
        "episodes_n": int(len(ep_local)),
        "agree_count": int(agree.sum()),
        "disagree_count": int((~agree).sum()),
        "agreement_rate_pct": float(100.0 * agree.mean()) if len(ep_local) else np.nan,
    })

EPISODE_REVIEWER_AGREEMENT_BY_MODEL = pd.DataFrame(agreement_rows).set_index("model").reindex(model_order).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.2))

# Agreement rate bar chart.
x = np.arange(len(model_order))
rates = EPISODE_REVIEWER_AGREEMENT_BY_MODEL["agreement_rate_pct"].values.astype(float)
axes[0].bar(x, rates, color=["#4E79A7", "#F28E2B", "#59A14F"][: len(model_order)])
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_order, rotation=15, ha="right")
axes[0].set_ylim(0, 105)
axes[0].set_ylabel("Agreement Rate (%)")
axes[0].set_title("Episode-Level Reviewer Agreement Rate")
axes[0].grid(axis="y", linestyle="--", alpha=0.3)

for i, v in enumerate(rates):
    axes[0].text(i, min(103, v + 2), f"{v:.1f}%", ha="center", va="bottom", fontsize=10)

# Agreement/disagreement stacked counts.
agree_counts = EPISODE_REVIEWER_AGREEMENT_BY_MODEL["agree_count"].values.astype(float)
disagree_counts = EPISODE_REVIEWER_AGREEMENT_BY_MODEL["disagree_count"].values.astype(float)
axes[1].bar(x, agree_counts, label="agree", color="#76B7B2")
axes[1].bar(x, disagree_counts, bottom=agree_counts, label="disagree", color="#E15759")
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_order, rotation=15, ha="right")
axes[1].set_ylabel("Episode Count")
axes[1].set_title("Episode-Level Agreement Counts")
axes[1].grid(axis="y", linestyle="--", alpha=0.3)

# Move legend off the chart area.
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle("Episode-Level Reviewer Agreement by Model", fontsize=17)
plt.tight_layout(rect=[0, 0, 0.93, 0.95])
plt.show()

## Reviewer Breakdown A: Agreement and Scenario Diagnostics

The agreement and scenario diagnostics show that reviewer-model alignment varied substantially by model and task. Overall completion agreement was lowest for qwen3, higher for qwen3.5, and intermediate for qwen3.5 mod1. This supports the broader pattern that qwen3 was too conservative, qwen3.5 was too permissive, and mod1 partially moderated qwen3.5’s completion-positive bias.

The scenario-level heatmaps show that qwen3.5 performed strongly when recall was most important, but its specificity was often weak. For example, in Plate-to-plate, qwen3.5 achieved high completion recall but predicted completion for every node, resulting in no true-negative completion cases. Qwen3.5 mod1 improved balance most clearly in Can-to-drawer-close, where it achieved strong precision, recall, specificity, and F1 relative to the other models. However, mod1 became more conservative in Cuttingboard-to-basket and Plate-to-plate, increasing missed completions compared with qwen3.5.

The episode-level reviewer comparison shows that node-level behavior does not always translate directly into accurate whole-episode success prediction. Qwen3.5 had the strongest episode-level F1 among the models because it captured most reviewer-success episodes, but this came with weaker specificity. Qwen3.5 mod1 reduced false positives but also missed more reviewer-success episodes. This reinforces the need to evaluate both node-level and episode-level agreement.

## Reviewer Breakdown B: Node Types and Difficulty

This section assigns each node to a role/type bucket, then summarizes where models struggle most.

Node type buckets:
- Navigation and positioning
- Grasp and acquire
- Place and handoff
- State change or interaction
- Perception and other

Tables in this section:
- Node-type by model breakdown
- Per-node difficulty ranking
- Node-type difficulty ranking

In [ ]:
if "NODE_REVIEW_DF" not in globals() or NODE_REVIEW_DF.empty:
    raise ValueError("NODE_REVIEW_DF is missing or empty. Run the reviewer-truth audit cell first.")
if "MODEL_REVIEW_SPECS" not in globals() or not MODEL_REVIEW_SPECS:
    raise ValueError("MODEL_REVIEW_SPECS is missing. Run the reviewer-truth audit cell first.")


def _node_type_from_id(node_id: object) -> str:
    s = str(node_id).strip().lower()
    if s == "" or s == "nan":
        return "Perception and other"

    if any(k in s for k in ["nav", "move", "goto", "go_to", "approach", "align", "position"]):
        return "Navigation and positioning"
    if any(k in s for k in ["grasp", "pick", "pickup", "close_gripper", "acquire", "grab"]):
        return "Grasp and acquire"
    if any(k in s for k in ["place", "put", "drop", "handoff", "release", "insert"]):
        return "Place and handoff"
    if any(k in s for k in ["open", "close", "toggle", "turn", "press", "activate", "deactivate"]):
        return "State change or interaction"
    return "Perception and other"


NODE_REVIEW_DF = NODE_REVIEW_DF.copy()
NODE_REVIEW_DF["node_type"] = NODE_REVIEW_DF["node_id"].map(_node_type_from_id)

model_frames = []
for spec in MODEL_REVIEW_SPECS:
    model_frames.append(
        NODE_REVIEW_DF.assign(
            model=spec["label"],
            model_complete=NODE_REVIEW_DF[f"{spec['key']}_complete"],
            model_failed=NODE_REVIEW_DF[f"{spec['key']}_failed"],
        )
    )
LONG_BY_MODEL = pd.concat(model_frames, ignore_index=True)

type_rows = []
for (node_type, model_label), g in LONG_BY_MODEL.groupby(["node_type", "model"], dropna=False):
    comp_agree = (g["reviewer_complete"].astype(bool) == g["model_complete"].astype(bool)).mean()
    fail_agree = (g["reviewer_failed"].astype(bool) == g["model_failed"].astype(bool)).mean()
    type_rows.append({
        "node_type": node_type,
        "model": model_label,
        "n": int(len(g)),
        "reviewer_completion_rate_pct": round(100.0 * g["reviewer_complete"].mean(), 1),
        "model_completion_rate_pct": round(100.0 * g["model_complete"].mean(), 1),
        "completion_agreement_pct": round(100.0 * comp_agree, 1),
        "reviewer_failure_rate_pct": round(100.0 * g["reviewer_failed"].mean(), 1),
        "model_failure_rate_pct": round(100.0 * g["model_failed"].mean(), 1),
        "failure_agreement_pct": round(100.0 * fail_agree, 1),
    })

NODE_TYPE_BREAKDOWN = pd.DataFrame(type_rows).sort_values(["node_type", "model"]).reset_index(drop=True)

difficulty_rows = []
for node_id, g in NODE_REVIEW_DF.groupby("node_id", dropna=False):
    per_model_disagree = []
    per_model_fn = []
    per_model_fp = []
    for spec in MODEL_REVIEW_SPECS:
        model_key = spec["key"]
        truth = g["reviewer_complete"].astype(bool)
        pred = g[f"{model_key}_complete"].astype(bool)
        per_model_disagree.append((truth != pred).mean())
        per_model_fn.append((truth & ~pred).mean())
        per_model_fp.append((~truth & pred).mean())

    difficulty_rows.append({
        "node_id": node_id,
        "node_type": g["node_type"].iloc[0],
        "n": int(len(g)),
        "reviewer_completion_rate_pct": round(100.0 * g["reviewer_complete"].mean(), 1),
        "avg_model_disagreement_pct": round(100.0 * float(np.mean(per_model_disagree)), 1),
        "avg_model_fn_pct": round(100.0 * float(np.mean(per_model_fn)), 1),
        "avg_model_fp_pct": round(100.0 * float(np.mean(per_model_fp)), 1),
        "difficulty_score": round(float(np.mean(per_model_disagree) + 0.5 * np.mean(per_model_fn)), 4),
    })

NODE_DIFFICULTY_BY_NODE = (
    pd.DataFrame(difficulty_rows)
    .sort_values(["difficulty_score", "avg_model_disagreement_pct", "n"], ascending=[False, False, False])
    .reset_index(drop=True)
)
NODE_DIFFICULTY_BY_NODE.insert(0, "difficulty_rank", np.arange(1, len(NODE_DIFFICULTY_BY_NODE) + 1))

NODE_DIFFICULTY_BY_TYPE = (
    NODE_DIFFICULTY_BY_NODE
    .groupby("node_type", dropna=False)
    .agg(
        node_types_n=("node_id", "nunique"),
        rows_n=("n", "sum"),
        mean_difficulty_score=("difficulty_score", "mean"),
        mean_disagreement_pct=("avg_model_disagreement_pct", "mean"),
        mean_fn_pct=("avg_model_fn_pct", "mean"),
        mean_fp_pct=("avg_model_fp_pct", "mean"),
    )
    .reset_index()
    .sort_values(["mean_difficulty_score", "mean_disagreement_pct"], ascending=False)
)

for c in ["mean_difficulty_score", "mean_disagreement_pct", "mean_fn_pct", "mean_fp_pct"]:
    NODE_DIFFICULTY_BY_TYPE[c] = NODE_DIFFICULTY_BY_TYPE[c].round(3 if c == "mean_difficulty_score" else 1)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(Markdown("### Node-Type or Node-Role Breakdown (Reviewer = Truth)"))
display(NODE_TYPE_BREAKDOWN)

display(Markdown("### Per-Node Difficulty Ranking"))
display(NODE_DIFFICULTY_BY_NODE)

display(Markdown("### Node-Type Difficulty Ranking"))
display(NODE_DIFFICULTY_BY_TYPE)

## Reviewer Breakdown B Visualizations

This section adds four visual diagnostics for node types and node difficulty:
1. Node-type x model performance heatmaps (completion and failure agreement).
2. Node-type error composition (reviewer-only vs model-only error rates).
3. Hardest nodes ranked by average disagreement rate.
4. Node support and coverage view (sample count vs disagreement).

In [ ]:
import matplotlib.pyplot as plt

if "NODE_TYPE_BREAKDOWN" not in globals() or NODE_TYPE_BREAKDOWN.empty:
    raise ValueError("NODE_TYPE_BREAKDOWN is missing or empty. Run Reviewer Breakdown B tables first.")
if "NODE_DIFFICULTY_BY_NODE" not in globals() or NODE_DIFFICULTY_BY_NODE.empty:
    raise ValueError("NODE_DIFFICULTY_BY_NODE is missing or empty. Run Reviewer Breakdown B tables first.")
if "LONG_BY_MODEL" not in globals() or LONG_BY_MODEL.empty:
    raise ValueError("LONG_BY_MODEL is missing or empty. Run Reviewer Breakdown B tables first.")
if "MODEL_REVIEW_SPECS" not in globals() or not MODEL_REVIEW_SPECS:
    raise ValueError("MODEL_REVIEW_SPECS is missing. Run reviewer audit tables first.")

model_order = [spec["label"] for spec in MODEL_REVIEW_SPECS]
node_type_order = (
    NODE_TYPE_BREAKDOWN.groupby("node_type", dropna=False)["n"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

# Shared node-type color map used in category-aware charts.
node_type_color_map = {
    nt: plt.cm.Set2(i / max(len(node_type_order), 1))
    for i, nt in enumerate(node_type_order)
}

# 1) Node-type x model performance heatmaps.
comp_heat = (
    NODE_TYPE_BREAKDOWN.pivot(index="node_type", columns="model", values="completion_agreement_pct")
    .reindex(index=node_type_order, columns=model_order)
)
fail_heat = (
    NODE_TYPE_BREAKDOWN.pivot(index="node_type", columns="model", values="failure_agreement_pct")
    .reindex(index=node_type_order, columns=model_order)
)

all_vals = np.concatenate([comp_heat.values.flatten(), fail_heat.values.flatten()])
all_vals = all_vals[np.isfinite(all_vals)]
vmin = float(all_vals.min()) if all_vals.size else 0.0
vmax = float(all_vals.max()) if all_vals.size else 100.0
if np.isclose(vmin, vmax):
    vmin = max(0.0, vmin - 5.0)
    vmax = min(100.0, vmax + 5.0)

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8), sharey=True, constrained_layout=True)
for ax, mat_df, title in [
    (axes[0], comp_heat, "Completion Agreement (%)"),
    (axes[1], fail_heat, "Failure Agreement (%)"),
]:
    mat = mat_df.values.astype(float)
    im = ax.imshow(mat, cmap="YlGnBu", vmin=vmin, vmax=vmax)
    ax.set_xticks(np.arange(len(model_order)))
    ax.set_yticks(np.arange(len(node_type_order)))
    ax.set_xticklabels(model_order, rotation=20, ha="right")
    ax.set_yticklabels(node_type_order)
    ax.set_title(title)
    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            v = mat[r, c]
            txt = "nan" if not np.isfinite(v) else f"{v:.1f}"
            ax.text(c, r, txt, ha="center", va="center", fontsize=9)

cbar = fig.colorbar(im, ax=axes, shrink=0.92, pad=0.02)
cbar.set_label("Agreement Rate (%)")
fig.suptitle("Reviewer Breakdown B: Node-Type x Model Performance", fontsize=17)
plt.show()

# 2) Node-type error composition: reviewer-only (FN) vs model-only (FP).
err_rows = []
for (node_type, model_label), g in LONG_BY_MODEL.groupby(["node_type", "model"], dropna=False):
    truth = g["reviewer_complete"].astype(bool)
    pred = g["model_complete"].astype(bool)
    n = int(len(g))
    reviewer_only_fn = int((truth & ~pred).sum())
    model_only_fp = int((~truth & pred).sum())
    err_rows.append({
        "node_type": node_type,
        "model": model_label,
        "n": n,
        "reviewer_only_rate_pct": (100.0 * reviewer_only_fn / n) if n else np.nan,
        "model_only_rate_pct": (100.0 * model_only_fp / n) if n else np.nan,
    })

err_df = pd.DataFrame(err_rows)

fig, axes = plt.subplots(1, len(model_order), figsize=(6.0 * len(model_order), 5.4), sharey=True)
if len(model_order) == 1:
    axes = [axes]

for ax, m in zip(axes, model_order):
    g = err_df[err_df["model"] == m].set_index("node_type").reindex(node_type_order)
    x = np.arange(len(node_type_order))
    fn_vals = g["reviewer_only_rate_pct"].values.astype(float)
    fp_vals = g["model_only_rate_pct"].values.astype(float)

    ax.bar(x, fn_vals, label="reviewer_only (FN)", color="#4E79A7")
    ax.bar(x, fp_vals, bottom=fn_vals, label="model_only (FP)", color="#E15759")
    ax.set_xticks(x)
    ax.set_xticklabels(node_type_order, rotation=25, ha="right")
    ax.set_title(m)
    ax.set_ylabel("Error composition rate (%)")
    ax.grid(axis="y", linestyle="--", alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle("Node-Type Error Composition (Completion Event)", fontsize=17)
plt.tight_layout(rect=[0, 0, 0.92, 0.95])
plt.show()

# 3) Hardest nodes ranked by disagreement rate, colored by node-type category.
top_n = min(15, len(NODE_DIFFICULTY_BY_NODE))
hard_nodes = NODE_DIFFICULTY_BY_NODE.head(top_n).copy()
hard_nodes = hard_nodes.sort_values("avg_model_disagreement_pct", ascending=True)

bar_colors = hard_nodes["node_type"].map(node_type_color_map)
y_labels = [f"{nid} [{nt}]" for nid, nt in zip(hard_nodes["node_id"], hard_nodes["node_type"])]

fig, ax = plt.subplots(figsize=(12.0, 7.0))
y = np.arange(len(hard_nodes))
ax.barh(y, hard_nodes["avg_model_disagreement_pct"].values.astype(float), color=bar_colors)
ax.set_yticks(y)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Average model disagreement (%)")
ax.set_title(f"Hardest Nodes by Disagreement Rate (Top {top_n}, Category-Aware)")
ax.grid(axis="x", linestyle="--", alpha=0.3)

for i, (v, n_support) in enumerate(zip(hard_nodes["avg_model_disagreement_pct"], hard_nodes["n"])):
    ax.text(float(v) + 0.6, i, f"n={int(n_support)}", va="center", fontsize=9)

legend_handles = [
    plt.Line2D([0], [0], marker="s", color="none", markerfacecolor=node_type_color_map[nt], markersize=10, label=nt)
    for nt in node_type_order
]
ax.legend(handles=legend_handles, loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, title="Node type")

plt.tight_layout(rect=[0, 0, 0.80, 1])
plt.show()

# 4) Node support / coverage: support count vs disagreement, colored by node type.
coverage_df = NODE_DIFFICULTY_BY_NODE.copy()

fig, ax = plt.subplots(figsize=(10.5, 6.0))
for nt in node_type_order:
    g = coverage_df[coverage_df["node_type"].astype(str) == nt]
    if g.empty:
        continue
    ax.scatter(
        g["n"].astype(float),
        g["avg_model_disagreement_pct"].astype(float),
        s=55,
        alpha=0.85,
        color=node_type_color_map[nt],
        label=nt,
        edgecolors="white",
        linewidths=0.5,
    )

ax.set_xlabel("Node support (reviewed rows)")
ax.set_ylabel("Average model disagreement (%)")
ax.set_title("Node Support / Coverage vs Disagreement")
ax.grid(True, linestyle="--", alpha=0.25)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)

plt.tight_layout(rect=[0, 0, 0.80, 1])
plt.show()

In [ ]:
if "NODE_DIFFICULTY_BY_TYPE" not in globals() or NODE_DIFFICULTY_BY_TYPE.empty:
    raise ValueError("NODE_DIFFICULTY_BY_TYPE is missing or empty. Run Reviewer Breakdown B tables first.")

summary_df = NODE_DIFFICULTY_BY_TYPE.copy()
summary_df = summary_df.sort_values("mean_difficulty_score", ascending=False).reset_index(drop=True)
summary_df["difficulty_rank"] = np.arange(1, len(summary_df) + 1)

# Use the same node-type palette from the previous chart if available.
if "node_type_color_map" in globals() and isinstance(node_type_color_map, dict):
    rank_colors = [node_type_color_map.get(nt, "#4E79A7") for nt in summary_df["node_type"]]
else:
    rank_colors = plt.cm.Set2(np.linspace(0, 1, len(summary_df)))

fig, ax = plt.subplots(figsize=(9.5, 5.8))
y = np.arange(len(summary_df))
ax.barh(y, summary_df["mean_difficulty_score"].astype(float), color=rank_colors)
ax.invert_yaxis()

rank_labels = [f"#{r} {nt}" for r, nt in zip(summary_df["difficulty_rank"], summary_df["node_type"])]
ax.set_yticks(y)
ax.set_yticklabels(rank_labels)
ax.set_xlabel("Mean difficulty score")
ax.set_title("Node-Type Ranking Summary")
ax.grid(axis="x", linestyle="--", alpha=0.3)

for i, (score, disag, rows_n) in enumerate(
    zip(
        summary_df["mean_difficulty_score"],
        summary_df["mean_disagreement_pct"],
        summary_df["rows_n"],
    )
):
    ax.text(
        float(score) + 0.005,
        i,
        f"disagree={float(disag):.1f}% | n={int(rows_n)}",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

## Reviewer Breakdown B: Node Types and Difficulty

The node-type breakdown shows that model performance depended on the type of BT subgoal being evaluated. Grasp-and-acquire nodes were generally easier for the qwen3.5-based models, with qwen3.5 and mod1 showing stronger completion agreement than qwen3. Place-and-handoff nodes were more difficult overall and accounted for many of the highest-disagreement nodes. State-change or interaction nodes also showed meaningful disagreement, especially where visual confirmation of the final state was more ambiguous.

The node difficulty rankings show that several of the hardest nodes involved placement or handoff actions, including `action_place_food_in_basket`, `node_action_place_lemon`, `node_action_place_food_in_basket`, and `node_action_place_food`. These nodes had high average disagreement and were often dominated by missed completions rather than false completions. This suggests that models may struggle most when the node requires judging whether an object has been correctly placed, transferred, or stabilized in the target location.

The node-type support plot is important for interpretation. Grasp-and-acquire and place-and-handoff had the most reviewed examples, while perception/other had only a small sample. Therefore, conclusions about grasp and placement nodes are more stable, while perception/other findings should be treated cautiously. Overall, the node-type analysis adds value by showing not just whether a model agrees with the reviewer, but where in the BT structure the disagreement occurs.


## Reviewer Breakdown C: Timing Alignment, Disagreement Types, and Simulator Reference

This section expands timing analysis into categorical alignment buckets and disagreement typologies.

Timing buckets (model minus reviewer):
- Early: less than -2 seconds
- On time: between -2 and +2 seconds
- Late: greater than +2 seconds

Tables in this section:
- Timing-category summary (completion and failure)
- Scenario-by-timing alignment
- Node-level timing alignment
- Disagreement-type summaries
- Simulator vs reviewer reference (reviewer = truth)

In [ ]:
if "NODE_REVIEW_DF" not in globals() or NODE_REVIEW_DF.empty:
    raise ValueError("NODE_REVIEW_DF is missing or empty. Run the reviewer-truth audit cell first.")
if "MODEL_REVIEW_SPECS" not in globals() or not MODEL_REVIEW_SPECS:
    raise ValueError("MODEL_REVIEW_SPECS is missing. Run the reviewer-truth audit cell first.")


def _timing_bucket(delta_s: float) -> str:
    if pd.isna(delta_s):
        return "unknown"
    if delta_s < -2.0:
        return "early_gt_2s"
    if delta_s > 2.0:
        return "late_gt_2s"
    return "on_time_pm_2s"


def _short_scenario_name_local(s: object) -> str:
    sc = str(s)
    if "SHORT_SCENARIO_MAP" in globals() and isinstance(SHORT_SCENARIO_MAP, dict) and sc in SHORT_SCENARIO_MAP:
        return str(SHORT_SCENARIO_MAP[sc])
    txt = sc.split("/")[-1]
    txt = txt.replace("_GR1ArmsAndWaistFourierHands_Env", "")
    txt = txt.replace("SplitA", "")
    txt = txt.replace("Posttrain", "")
    txt = txt.replace("PnP", "")
    txt = txt.replace("_", " ").strip()
    return txt if txt else sc


def _conf_breakdown(truth: pd.Series, pred: pd.Series) -> dict[str, float]:
    t = truth.astype(bool)
    p = pred.astype(bool)
    n = int(len(t))
    tp = int((t & p).sum())
    tn = int((~t & ~p).sum())
    fn = int((t & ~p).sum())
    fp = int((~t & p).sum())
    return {
        "n": n,
        "both_event_tp": tp,
        "both_no_event_tn": tn,
        "reviewer_only_fn": fn,
        "model_only_fp": fp,
        "both_event_pct": round(100.0 * tp / n, 1) if n else np.nan,
        "both_no_event_pct": round(100.0 * tn / n, 1) if n else np.nan,
        "reviewer_only_pct": round(100.0 * fn / n, 1) if n else np.nan,
        "model_only_pct": round(100.0 * fp / n, 1) if n else np.nan,
        "disagreement_pct": round(100.0 * (fn + fp) / n, 1) if n else np.nan,
    }


timing_rows = []
for spec in MODEL_REVIEW_SPECS:
    model_key = spec["key"]
    model_label = spec["label"]

    for event_type, reviewer_col, model_col in [
        ("completion", "reviewer_completion_s", f"{model_key}_completion_s"),
        ("failure", "reviewer_failure_s", f"{model_key}_failure_s"),
    ]:
        valid = NODE_REVIEW_DF[["scenario_id", "node_id", "node_index", reviewer_col, model_col]].dropna().copy()
        if valid.empty:
            continue
        valid["model"] = model_label
        valid["event_type"] = event_type
        valid["delta_s"] = valid[model_col] - valid[reviewer_col]
        valid["timing_bucket"] = valid["delta_s"].map(_timing_bucket)
        timing_rows.append(valid)

TIMING_ALIGNMENT_LONG = pd.concat(timing_rows, ignore_index=True) if timing_rows else pd.DataFrame(columns=["scenario_id", "node_id", "node_index", "model", "event_type", "delta_s", "timing_bucket"])

if not TIMING_ALIGNMENT_LONG.empty:
    TIMING_ALIGNMENT_LONG["short_scenario"] = TIMING_ALIGNMENT_LONG["scenario_id"].map(_short_scenario_name_local)

if "node_type" in NODE_REVIEW_DF.columns and not TIMING_ALIGNMENT_LONG.empty:
    node_types = NODE_REVIEW_DF[["node_id", "node_type"]].drop_duplicates()
    TIMING_ALIGNMENT_LONG = TIMING_ALIGNMENT_LONG.merge(node_types, on="node_id", how="left")


TIMING_ALIGNMENT_CATEGORY_SUMMARY = (
    TIMING_ALIGNMENT_LONG
    .groupby(["model", "event_type", "timing_bucket"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
if not TIMING_ALIGNMENT_CATEGORY_SUMMARY.empty:
    totals = TIMING_ALIGNMENT_CATEGORY_SUMMARY.groupby(["model", "event_type"])["count"].transform("sum")
    TIMING_ALIGNMENT_CATEGORY_SUMMARY["pct"] = (100.0 * TIMING_ALIGNMENT_CATEGORY_SUMMARY["count"] / totals).round(1)


TIMING_ALIGNMENT_BY_SCENARIO = (
    TIMING_ALIGNMENT_LONG
    .groupby(["short_scenario", "model", "event_type", "timing_bucket"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
if not TIMING_ALIGNMENT_BY_SCENARIO.empty:
    totals_sc = TIMING_ALIGNMENT_BY_SCENARIO.groupby(["short_scenario", "model", "event_type"])["count"].transform("sum")
    TIMING_ALIGNMENT_BY_SCENARIO["pct"] = (100.0 * TIMING_ALIGNMENT_BY_SCENARIO["count"] / totals_sc).round(1)


node_group_cols = ["node_id", "model", "event_type", "timing_bucket"]
if "node_type" in TIMING_ALIGNMENT_LONG.columns:
    node_group_cols = ["node_id", "node_type", "model", "event_type", "timing_bucket"]

TIMING_ALIGNMENT_BY_NODE = (
    TIMING_ALIGNMENT_LONG
    .groupby(node_group_cols, dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
if not TIMING_ALIGNMENT_BY_NODE.empty:
    base_cols = ["node_id", "model", "event_type"]
    if "node_type" in TIMING_ALIGNMENT_BY_NODE.columns:
        base_cols = ["node_id", "node_type", "model", "event_type"]
    totals_node = TIMING_ALIGNMENT_BY_NODE.groupby(base_cols)["count"].transform("sum")
    TIMING_ALIGNMENT_BY_NODE["pct"] = (100.0 * TIMING_ALIGNMENT_BY_NODE["count"] / totals_node).round(1)

NODE_TIMING_ALIGNMENT_SUMMARY = (
    TIMING_ALIGNMENT_LONG
    .groupby(["node_id", "model", "event_type"], dropna=False)
    .agg(
        pairs_n=("delta_s", "size"),
        mean_delta_s=("delta_s", "mean"),
        median_delta_s=("delta_s", "median"),
        mae_s=("delta_s", lambda s: float(np.mean(np.abs(s)))),
        pct_on_time=("timing_bucket", lambda s: float(100.0 * (s == "on_time_pm_2s").mean())),
        pct_early=("timing_bucket", lambda s: float(100.0 * (s == "early_gt_2s").mean())),
        pct_late=("timing_bucket", lambda s: float(100.0 * (s == "late_gt_2s").mean())),
    )
    .reset_index()
)
if "node_type" in TIMING_ALIGNMENT_LONG.columns and not NODE_TIMING_ALIGNMENT_SUMMARY.empty:
    node_type_map = TIMING_ALIGNMENT_LONG[["node_id", "node_type"]].drop_duplicates()
    NODE_TIMING_ALIGNMENT_SUMMARY = NODE_TIMING_ALIGNMENT_SUMMARY.merge(node_type_map, on="node_id", how="left")

for col in ["mean_delta_s", "median_delta_s", "mae_s", "pct_on_time", "pct_early", "pct_late"]:
    if col in NODE_TIMING_ALIGNMENT_SUMMARY.columns:
        NODE_TIMING_ALIGNMENT_SUMMARY[col] = NODE_TIMING_ALIGNMENT_SUMMARY[col].round(3 if "delta" in col or "mae" in col else 1)


disagree_rows = []
for spec in MODEL_REVIEW_SPECS:
    model_key = spec["key"]
    model_label = spec["label"]
    for event_type, reviewer_col, model_col in [
        ("completion", "reviewer_complete", f"{model_key}_complete"),
        ("failure", "reviewer_failed", f"{model_key}_failed"),
    ]:
        disagree_rows.append({
            "model": model_label,
            "event_type": event_type,
            **_conf_breakdown(NODE_REVIEW_DF[reviewer_col], NODE_REVIEW_DF[model_col]),
        })

DISAGREEMENT_TYPE_SUMMARY = pd.DataFrame(disagree_rows).sort_values(["event_type", "model"]).reset_index(drop=True)


if "ep" in globals() and isinstance(ep, pd.DataFrame) and not ep.empty and {"scenario_id", "reviewer_episode_success", "sim_episode_success"}.issubset(ep.columns):
    _ep_ref = ep.copy()
elif "reviewed_eps" in globals() and isinstance(reviewed_eps, pd.DataFrame) and not reviewed_eps.empty:
    if "_is_filled" not in globals() or "_to_bool" not in globals():
        raise ValueError("Helper functions are missing. Re-run the reviewer-truth audit cell first.")

    def _episode_truth_from_row_local(row: pd.Series, comp_suffix: str) -> bool:
        n_nodes = max(0, int(pd.to_numeric(row.get("bt_action_nodes", 0), errors="coerce") if pd.notna(row.get("bt_action_nodes", np.nan)) else 0))
        if n_nodes == 0:
            return False
        for i in range(1, n_nodes + 1):
            if not _is_filled(row.get(f"node_{i}_{comp_suffix}", np.nan)):
                return False
        return True

    _ep_ref = reviewed_eps.copy()
    _ep_ref["reviewer_episode_success"] = _ep_ref.apply(lambda r: _episode_truth_from_row_local(r, "reviewer_completion_time"), axis=1)
    _ep_ref["sim_episode_success"] = _ep_ref["success"].map(_to_bool)
else:
    _ep_ref = pd.DataFrame()


def _conf_episode(truth: pd.Series, pred: pd.Series) -> dict[str, float]:
    t = truth.astype(bool)
    p = pred.astype(bool)
    tp = int((t & p).sum())
    fp = int((~t & p).sum())
    tn = int((~t & ~p).sum())
    fn = int((t & ~p).sum())
    n = int(len(t))
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    accuracy = (tp + tn) / n if n else np.nan
    f1 = (2 * precision * recall / (precision + recall)) if (pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0) else np.nan
    return {
        "n": n,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1,
    }

if _ep_ref.empty:
    SIMULATOR_VS_REVIEWER_REFERENCE_OVERALL = pd.DataFrame([
        {
            "predictor": "simulator_success",
            "reviewer_success_rate_pct": np.nan,
            "predictor_success_rate_pct": np.nan,
            "n": 0,
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
            "precision": np.nan,
            "recall": np.nan,
            "specificity": np.nan,
            "accuracy": np.nan,
            "f1": np.nan,
        }
    ])
    SIMULATOR_VS_REVIEWER_REFERENCE_SCENARIO = pd.DataFrame()
else:
    overall_conf = _conf_episode(_ep_ref["reviewer_episode_success"], _ep_ref["sim_episode_success"])
    SIMULATOR_VS_REVIEWER_REFERENCE_OVERALL = pd.DataFrame([
        {
            "predictor": "simulator_success",
            "reviewer_success_rate_pct": round(100.0 * _ep_ref["reviewer_episode_success"].mean(), 1),
            "predictor_success_rate_pct": round(100.0 * _ep_ref["sim_episode_success"].mean(), 1),
            **overall_conf,
        }
    ])

    scenario_rows = []
    for scenario_id, g in _ep_ref.groupby("scenario_id", dropna=False):
        conf = _conf_episode(g["reviewer_episode_success"], g["sim_episode_success"])
        scenario_rows.append({
            "short_scenario": _short_scenario_name_local(scenario_id),
            "predictor": "simulator_success",
            "reviewer_success_rate_pct": round(100.0 * g["reviewer_episode_success"].mean(), 1),
            "predictor_success_rate_pct": round(100.0 * g["sim_episode_success"].mean(), 1),
            **conf,
        })
    SIMULATOR_VS_REVIEWER_REFERENCE_SCENARIO = pd.DataFrame(scenario_rows).sort_values("short_scenario").reset_index(drop=True)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(Markdown("### Timing-Category Summary (Early / On Time / Late)"))
display(TIMING_ALIGNMENT_CATEGORY_SUMMARY)

display(Markdown("### Scenario-by-Timing Alignment"))
display(TIMING_ALIGNMENT_BY_SCENARIO)

display(Markdown("### Node-Level Timing Alignment (Bucket Distribution)"))
display(TIMING_ALIGNMENT_BY_NODE)

display(Markdown("### Node-Level Timing Alignment (Delta Summary)"))
display(NODE_TIMING_ALIGNMENT_SUMMARY)

display(Markdown("### Disagreement-Type Summaries"))
display(DISAGREEMENT_TYPE_SUMMARY)

display(Markdown("### Simulator vs Reviewer Reference (Reviewer = Truth): Overall"))
display(SIMULATOR_VS_REVIEWER_REFERENCE_OVERALL)

display(Markdown("### Simulator vs Reviewer Reference (Reviewer = Truth): By Scenario"))
display(SIMULATOR_VS_REVIEWER_REFERENCE_SCENARIO)

## Reviewer Breakdown C Visualizations

This section adds eight visual diagnostics for timing alignment, disagreement types, and simulator reference.

Figures in this section:
1. Timing delta distribution by model.
2. Early / on-time / late timing classification.
3. Scenario x model timing heatmap.
4. Node-type timing heatmap.
5. Disagreement by scenario.
6. Disagreement-type stacked bar chart.
7. Simulator vs reviewer confusion matrix.
8. Reference comparison bar chart (model and simulator agreement with reviewer at episode level).

In [ ]:
import matplotlib.pyplot as plt

if "TIMING_ALIGNMENT_LONG" not in globals() or TIMING_ALIGNMENT_LONG.empty:
    raise ValueError("TIMING_ALIGNMENT_LONG is missing or empty. Run Reviewer Breakdown C tables first.")
if "TIMING_ALIGNMENT_CATEGORY_SUMMARY" not in globals() or TIMING_ALIGNMENT_CATEGORY_SUMMARY.empty:
    raise ValueError("TIMING_ALIGNMENT_CATEGORY_SUMMARY is missing or empty. Run Reviewer Breakdown C tables first.")
if "TIMING_ALIGNMENT_BY_SCENARIO" not in globals() or TIMING_ALIGNMENT_BY_SCENARIO.empty:
    raise ValueError("TIMING_ALIGNMENT_BY_SCENARIO is missing or empty. Run Reviewer Breakdown C tables first.")
if "DISAGREEMENT_TYPE_SUMMARY" not in globals() or DISAGREEMENT_TYPE_SUMMARY.empty:
    raise ValueError("DISAGREEMENT_TYPE_SUMMARY is missing or empty. Run Reviewer Breakdown C tables first.")
if "SIMULATOR_VS_REVIEWER_REFERENCE_OVERALL" not in globals() or SIMULATOR_VS_REVIEWER_REFERENCE_OVERALL.empty:
    raise ValueError("SIMULATOR_VS_REVIEWER_REFERENCE_OVERALL is missing or empty. Run Reviewer Breakdown C tables first.")
if "NODE_REVIEW_DF" not in globals() or NODE_REVIEW_DF.empty:
    raise ValueError("NODE_REVIEW_DF is missing or empty. Run the reviewer-truth audit cell first.")
if "MODEL_REVIEW_SPECS" not in globals() or not MODEL_REVIEW_SPECS:
    raise ValueError("MODEL_REVIEW_SPECS is missing. Run the reviewer-truth audit cell first.")

model_order = [spec["label"] for spec in MODEL_REVIEW_SPECS]
event_order = ["completion", "failure"]
event_label = {"completion": "Completion", "failure": "Failure"}
bucket_order = ["early_gt_2s", "on_time_pm_2s", "late_gt_2s"]
bucket_label = {
    "early_gt_2s": "Early (< -2s)",
    "on_time_pm_2s": "On time (+/-2s)",
    "late_gt_2s": "Late (> +2s)",
}

def _short_scenario_name_vis(s: object) -> str:
    sc = str(s)
    if "SHORT_SCENARIO_MAP" in globals() and isinstance(SHORT_SCENARIO_MAP, dict) and sc in SHORT_SCENARIO_MAP:
        return str(SHORT_SCENARIO_MAP[sc])
    txt = sc.split("/")[-1]
    txt = txt.replace("_GR1ArmsAndWaistFourierHands_Env", "")
    txt = txt.replace("SplitA", "")
    txt = txt.replace("Posttrain", "")
    txt = txt.replace("PnP", "")
    txt = txt.replace("_", " ").strip()
    return txt if txt else sc

# 1) Timing delta distribution by model (violin chart for completion/failure).
delta_df = TIMING_ALIGNMENT_LONG[TIMING_ALIGNMENT_LONG["event_type"].isin(event_order)].copy()
fig, axes = plt.subplots(1, len(model_order), figsize=(5.7 * len(model_order), 5.0), sharey=True)
if len(model_order) == 1:
    axes = [axes]

for ax, model_label in zip(axes, model_order):
    g = delta_df[delta_df["model"].astype(str) == model_label]
    comp_vals = g[g["event_type"] == "completion"]["delta_s"].dropna().astype(float).values
    fail_vals = g[g["event_type"] == "failure"]["delta_s"].dropna().astype(float).values

    violin_data = []
    positions = []
    labels = []
    colors = []

    if len(comp_vals):
        violin_data.append(comp_vals)
        positions.append(1)
        labels.append("completion")
        colors.append("#4E79A7")
    if len(fail_vals):
        violin_data.append(fail_vals)
        positions.append(2)
        labels.append("failure")
        colors.append("#E15759")

    if not violin_data:
        ax.set_title(f"{model_label} (no timing pairs)")
        ax.set_xticks([1, 2])
        ax.set_xticklabels(["completion", "failure"], rotation=12, ha="right")
        continue

    parts = ax.violinplot(
        violin_data,
        positions=positions,
        widths=0.75,
        showmeans=True,
        showmedians=True,
        showextrema=False,
    )
    for body, color in zip(parts["bodies"], colors):
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_alpha(0.55)
    parts["cmeans"].set_color("black")
    parts["cmeans"].set_linewidth(1.2)
    parts["cmedians"].set_color("dimgray")
    parts["cmedians"].set_linewidth(1.2)

    for pos, vals in zip(positions, violin_data):
        ax.scatter(
            np.full(len(vals), pos, dtype=float),
            vals,
            s=10,
            alpha=0.18,
            color="black",
            zorder=3,
        )

    ax.axhline(0.0, color="black", linestyle="--", linewidth=1.1)
    ax.axhline(-2.0, color="gray", linestyle=":", linewidth=1.0)
    ax.axhline(2.0, color="gray", linestyle=":", linewidth=1.0)
    ax.set_xlim(0.4, 2.6)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["completion", "failure"], rotation=12, ha="right")
    ax.set_title(model_label)
    ax.set_xlabel("Event type")
    ax.grid(axis="y", linestyle="--", alpha=0.25)

all_delta_vals = delta_df["delta_s"].dropna().astype(float).values
if len(all_delta_vals):
    clip = np.nanpercentile(np.abs(all_delta_vals), 98)
    clip = max(3.0, float(clip))
    axes[0].set_ylim(-clip, clip)
axes[0].set_ylabel("Model - Reviewer timing delta (s)")
fig.suptitle("Timing Delta Distribution by Model (Violin)", fontsize=17)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# 2) Early / on-time / late classification (stacked bars).
class_df = TIMING_ALIGNMENT_CATEGORY_SUMMARY[
    TIMING_ALIGNMENT_CATEGORY_SUMMARY["timing_bucket"].isin(bucket_order)
].copy()

fig, axes = plt.subplots(1, len(event_order), figsize=(6.8 * len(event_order), 5.0), sharey=True)
if len(event_order) == 1:
    axes = [axes]

bucket_colors = {
    "early_gt_2s": "#E15759",
    "on_time_pm_2s": "#59A14F",
    "late_gt_2s": "#4E79A7",
}

for ax, ev in zip(axes, event_order):
    tbl = (
        class_df[class_df["event_type"] == ev]
        .pivot(index="model", columns="timing_bucket", values="pct")
        .reindex(index=model_order, columns=bucket_order)
        .fillna(0.0)
    )
    x = np.arange(len(model_order))
    bottom = np.zeros(len(model_order), dtype=float)

    for b in bucket_order:
        vals = tbl[b].values.astype(float)
        ax.bar(x, vals, bottom=bottom, label=bucket_label[b], color=bucket_colors[b])
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(model_order, rotation=15, ha="right")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Share of timing pairs (%)")
    ax.set_title(event_label[ev])
    ax.grid(axis="y", linestyle="--", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.03, 0.5), frameon=False)
fig.suptitle("Early / On-Time / Late Timing Classification", fontsize=17)
plt.tight_layout(rect=[0, 0, 0.90, 0.95])
plt.show()

# 3) Scenario x model timing heatmap (on-time %).
scenario_order = []
if "SCENARIO_ORDER" in globals() and isinstance(SCENARIO_ORDER, list):
    scenario_order = [s for s in SCENARIO_ORDER if s in TIMING_ALIGNMENT_BY_SCENARIO["short_scenario"].astype(str).unique().tolist()]
remaining = sorted(
    set(TIMING_ALIGNMENT_BY_SCENARIO["short_scenario"].astype(str).unique().tolist()) - set(scenario_order)
)
scenario_order = scenario_order + remaining

fig, axes = plt.subplots(1, len(event_order), figsize=(7.2 * len(event_order), 5.4), sharey=True, constrained_layout=True)
if len(event_order) == 1:
    axes = [axes]

last_im = None
for ax, ev in zip(axes, event_order):
    hm = (
        TIMING_ALIGNMENT_BY_SCENARIO[
            (TIMING_ALIGNMENT_BY_SCENARIO["event_type"] == ev)
            & (TIMING_ALIGNMENT_BY_SCENARIO["timing_bucket"] == "on_time_pm_2s")
        ]
        .pivot(index="short_scenario", columns="model", values="pct")
        .reindex(index=scenario_order, columns=model_order)
    )
    mat = hm.values.astype(float)
    im = ax.imshow(mat, cmap="YlGnBu", vmin=0, vmax=100)
    last_im = im
    ax.set_xticks(np.arange(len(model_order)))
    ax.set_yticks(np.arange(len(scenario_order)))
    ax.set_xticklabels(model_order, rotation=20, ha="right")
    ax.set_yticklabels(scenario_order)
    ax.set_title(f"{event_label[ev]} on-time %")

    for r in range(mat.shape[0]):
        for c in range(mat.shape[1]):
            v = mat[r, c]
            txt = "nan" if not np.isfinite(v) else f"{v:.1f}"
            ax.text(c, r, txt, ha="center", va="center", fontsize=8)

if last_im is not None:
    cbar = fig.colorbar(last_im, ax=axes, location="right", shrink=0.92, pad=0.02)
    cbar.set_label("On-time share (%)")
fig.suptitle("Scenario x Model Timing Heatmap", fontsize=17)
plt.show()

# 4) Node-type timing heatmap (on-time %).
if "node_type" in TIMING_ALIGNMENT_LONG.columns:
    node_type_summary = (
        TIMING_ALIGNMENT_LONG[TIMING_ALIGNMENT_LONG["timing_bucket"].isin(bucket_order)]
        .groupby(["node_type", "model", "event_type", "timing_bucket"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    totals = node_type_summary.groupby(["node_type", "model", "event_type"], dropna=False)["count"].transform("sum")
    node_type_summary["pct"] = 100.0 * node_type_summary["count"] / totals
    node_type_order = (
        node_type_summary.groupby("node_type", dropna=False)["count"]
        .sum()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    fig, axes = plt.subplots(1, len(event_order), figsize=(7.2 * len(event_order), 5.6), sharey=True, constrained_layout=True)
    if len(event_order) == 1:
        axes = [axes]

    last_im = None
    for ax, ev in zip(axes, event_order):
        hm = (
            node_type_summary[
                (node_type_summary["event_type"] == ev)
                & (node_type_summary["timing_bucket"] == "on_time_pm_2s")
            ]
            .pivot(index="node_type", columns="model", values="pct")
            .reindex(index=node_type_order, columns=model_order)
        )
        mat = hm.values.astype(float)
        im = ax.imshow(mat, cmap="YlOrBr", vmin=0, vmax=100)
        last_im = im
        ax.set_xticks(np.arange(len(model_order)))
        ax.set_yticks(np.arange(len(node_type_order)))
        ax.set_xticklabels(model_order, rotation=20, ha="right")
        ax.set_yticklabels(node_type_order)
        ax.set_title(f"{event_label[ev]} on-time %")

        for r in range(mat.shape[0]):
            for c in range(mat.shape[1]):
                v = mat[r, c]
                txt = "nan" if not np.isfinite(v) else f"{v:.1f}"
                ax.text(c, r, txt, ha="center", va="center", fontsize=8)

    if last_im is not None:
        cbar = fig.colorbar(last_im, ax=axes, location="right", shrink=0.92, pad=0.02)
        cbar.set_label("On-time share (%)")
    fig.suptitle("Node-Type Timing Heatmap", fontsize=17)
    plt.show()

# 5) Disagreement by scenario (completion and failure).
dis_rows = []
for scenario_id, g_sc in NODE_REVIEW_DF.groupby("scenario_id", dropna=False):
    sc_label = _short_scenario_name_vis(scenario_id)
    for spec in MODEL_REVIEW_SPECS:
        mk = spec["key"]
        for ev, truth_col, pred_col in [
            ("completion", "reviewer_complete", f"{mk}_complete"),
            ("failure", "reviewer_failed", f"{mk}_failed"),
        ]:
            truth = g_sc[truth_col].astype(bool)
            pred = g_sc[pred_col].astype(bool)
            n = int(len(g_sc))
            dis_rate = 100.0 * float((truth != pred).mean()) if n else np.nan
            dis_rows.append({
                "short_scenario": sc_label,
                "model": spec["label"],
                "event_type": ev,
                "disagreement_pct": dis_rate,
            })

dis_by_scenario = pd.DataFrame(dis_rows)
scenario_order2 = []
if "SCENARIO_ORDER" in globals() and isinstance(SCENARIO_ORDER, list):
    scenario_order2 = [s for s in SCENARIO_ORDER if s in dis_by_scenario["short_scenario"].astype(str).unique().tolist()]
remain2 = sorted(set(dis_by_scenario["short_scenario"].astype(str).unique().tolist()) - set(scenario_order2))
scenario_order2 = scenario_order2 + remain2

fig, axes = plt.subplots(1, len(event_order), figsize=(7.2 * len(event_order), 5.2), sharey=True)
if len(event_order) == 1:
    axes = [axes]

for ax, ev in zip(axes, event_order):
    tbl = (
        dis_by_scenario[dis_by_scenario["event_type"] == ev]
        .pivot(index="short_scenario", columns="model", values="disagreement_pct")
        .reindex(index=scenario_order2, columns=model_order)
    )

    x = np.arange(len(tbl.index))
    w = 0.8 / max(len(model_order), 1)
    for i, m in enumerate(model_order):
        offset = (i - (len(model_order) - 1) / 2.0) * w
        ax.bar(x + offset, tbl[m].values.astype(float), width=w, label=m)

    ax.set_xticks(x)
    ax.set_xticklabels(tbl.index, rotation=15, ha="right")
    ax.set_ylabel("Disagreement rate (%)")
    ax.set_title(event_label[ev])
    ax.grid(axis="y", linestyle="--", alpha=0.25)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.suptitle("Disagreement by Scenario", fontsize=17)
plt.tight_layout(rect=[0, 0, 0.92, 0.95])
plt.show()

# 6) Disagreement-type stacked bar chart.
stack_df = DISAGREEMENT_TYPE_SUMMARY.copy()
fig, axes = plt.subplots(1, len(event_order), figsize=(7.0 * len(event_order), 5.0), sharey=True)
if len(event_order) == 1:
    axes = [axes]

for ax, ev in zip(axes, event_order):
    g = stack_df[stack_df["event_type"] == ev].set_index("model").reindex(model_order)
    x = np.arange(len(model_order))
    both_event = g["both_event_pct"].fillna(0.0).values.astype(float)
    both_no = g["both_no_event_pct"].fillna(0.0).values.astype(float)
    rev_only = g["reviewer_only_pct"].fillna(0.0).values.astype(float)
    mod_only = g["model_only_pct"].fillna(0.0).values.astype(float)

    ax.bar(x, both_event, label="both event", color="#59A14F")
    ax.bar(x, both_no, bottom=both_event, label="both no-event", color="#9C755F")
    ax.bar(x, rev_only, bottom=both_event + both_no, label="reviewer only", color="#4E79A7")
    ax.bar(x, mod_only, bottom=both_event + both_no + rev_only, label="model only", color="#E15759")

    ax.set_xticks(x)
    ax.set_xticklabels(model_order, rotation=15, ha="right")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Share of evaluated nodes (%)")
    ax.set_title(event_label[ev])
    ax.grid(axis="y", linestyle="--", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center right", bbox_to_anchor=(1.03, 0.5), frameon=False)
fig.suptitle("Disagreement Type Composition", fontsize=17)
plt.tight_layout(rect=[0, 0, 0.90, 0.95])
plt.show()

# 7) Simulator vs reviewer confusion matrix (episode level, overall).
sim_row = SIMULATOR_VS_REVIEWER_REFERENCE_OVERALL.iloc[0]
sim_mat = np.array([[sim_row.get("tp", 0), sim_row.get("fn", 0)], [sim_row.get("fp", 0), sim_row.get("tn", 0)]], dtype=float)
vmax = np.nanmax(sim_mat) if np.isfinite(sim_mat).any() else 1.0
if not np.isfinite(vmax) or vmax <= 0:
    vmax = 1.0

fig, ax = plt.subplots(figsize=(5.6, 4.8))
ax.imshow(sim_mat, cmap="Blues", vmin=0, vmax=vmax)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred: Success", "Pred: Failure"], rotation=15, ha="right")
ax.set_yticklabels(["True: Success", "True: Failure"])
ax.set_title("Simulator vs Reviewer Confusion Matrix")

for r in range(2):
    for c in range(2):
        v = sim_mat[r, c]
        txt = "nan" if not np.isfinite(v) else str(int(v))
        color = "white" if np.isfinite(v) and v > (vmax / 2.0) else "black"
        ax.text(c, r, txt, ha="center", va="center", color=color, fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

# 8) Reference comparison: model and simulator agreement with reviewer (episode level).
ref_rows = []
if "EPISODE_VS_REVIEWER_SUMMARY" in globals() and isinstance(EPISODE_VS_REVIEWER_SUMMARY, pd.DataFrame) and not EPISODE_VS_REVIEWER_SUMMARY.empty:
    for _, r in EPISODE_VS_REVIEWER_SUMMARY.iterrows():
        if pd.notna(r.get("accuracy", np.nan)):
            ref_rows.append({
                "predictor": str(r.get("predictor", "unknown")),
                "agreement_rate_pct": 100.0 * float(r["accuracy"]),
            })

if pd.notna(sim_row.get("accuracy", np.nan)):
    ref_rows.append({
        "predictor": "simulator_success",
        "agreement_rate_pct": 100.0 * float(sim_row["accuracy"]),
    })

ref_df = pd.DataFrame(ref_rows)
if not ref_df.empty:
    ref_df = ref_df.drop_duplicates(subset=["predictor"], keep="last")
    desired_order = model_order + ["simulator_success"]
    ref_df["_ord"] = ref_df["predictor"].map({k: i for i, k in enumerate(desired_order)})
    ref_df = ref_df.sort_values(["_ord", "predictor"]).drop(columns=["_ord"]).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    x = np.arange(len(ref_df))
    colors = ["#4E79A7" if p != "simulator_success" else "#76B7B2" for p in ref_df["predictor"]]
    bars = ax.bar(x, ref_df["agreement_rate_pct"].values.astype(float), color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(ref_df["predictor"], rotation=15, ha="right")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Agreement with reviewer (%)")
    ax.set_title("Reference Comparison: Episode-Level Agreement with Reviewer")
    ax.grid(axis="y", linestyle="--", alpha=0.25)

    for b in bars:
        h = float(b.get_height())
        ax.text(b.get_x() + b.get_width() / 2, min(99.0, h + 1.5), f"{h:.1f}%", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()

## Reviewer Breakdown C: Timing Alignment, Disagreement Types, and Simulator Reference

The timing alignment results show that models often identified completion near the reviewer’s timestamp, but their timing bias differed. Qwen3 and qwen3.5 mod1 were usually within ±2 seconds of the reviewer for matched completion events, with 89.7% and 89.5% on-time rates, respectively. Qwen3.5 was also often close, with 81.7% on time, but it had a stronger early-completion pattern. This supports the interpretation that qwen3.5 was not only more likely to call completion, but also more likely to call it early.

The scenario-level timing breakdown shows that timing reliability varied by task. Qwen3.5 was especially early in Cuttingboard-to-basket, where 35.7% of matched completion calls occurred more than two seconds before the reviewer. In contrast, mod1 showed stronger timing alignment in Cuttingboard-to-basket and Plate-to-plate, with over 90% of matched completion calls within the ±2 second window. This suggests that prompt modification improved not only completion selectivity but also temporal alignment in some scenarios.

The disagreement-type summaries clarify the dominant error modes. For completion, qwen3 mainly produced reviewer-only errors, meaning the reviewer marked nodes complete but qwen3 did not. Qwen3.5 mainly produced model-only errors, meaning the model marked completion when the reviewer did not. Qwen3.5 mod1 reduced model-only errors relative to qwen3.5, but increased reviewer-only missed completions. For failure detection, qwen3.5 and mod1 missed all reviewer-labeled failures because they did not predict failure states.

Finally, the simulator-reference comparison showed strong alignment between simulator success and reviewer episode-level success. Simulator success achieved 98.3% accuracy and an F1 of 0.986 against reviewer labels, with only one false positive and no false negatives. This supports using simulator success as a reliable episode-level reference for the broader dataset, while the reviewer annotations provide additional node-level, timing, and disagreement detail that the simulator alone cannot capture.

## Mod1 Deep Dive: Reviewer-Missed Completion Flags (False Positives)

This section isolates reviewed-node instances where **qwen3.5 mod1 marked completion but the reviewer did not**.

It reports:
1. Node-type ranking for these false-positive completion events.
2. The top node-type where this happens most.
3. A representative sample of 4-5 episodes from that top node-type for manual inspection.

In [ ]:
if "NODE_REVIEW_DF" not in globals() or NODE_REVIEW_DF.empty:
    raise ValueError("NODE_REVIEW_DF is missing or empty. Run the reviewer-truth audit cell first.")
if "SHORT_SCENARIO_MAP" not in globals() or not isinstance(SHORT_SCENARIO_MAP, dict):
    SHORT_SCENARIO_MAP = {}

# Ensure we have node_type available (created in Breakdown B).
work_df = NODE_REVIEW_DF.copy()
if "node_type" not in work_df.columns:
    def _node_type_from_id_local(node_id: object) -> str:
        s = str(node_id).strip().lower()
        if s == "" or s == "nan":
            return "Perception and other"
        if any(k in s for k in ["nav", "move", "goto", "go_to", "approach", "align", "position"]):
            return "Navigation and positioning"
        if any(k in s for k in ["grasp", "pick", "pickup", "close_gripper", "acquire", "grab"]):
            return "Grasp and acquire"
        if any(k in s for k in ["place", "put", "drop", "handoff", "release", "insert"]):
            return "Place and handoff"
        if any(k in s for k in ["open", "close", "toggle", "turn", "press", "activate", "deactivate"]):
            return "State change or interaction"
        return "Perception and other"
    work_df["node_type"] = work_df["node_id"].map(_node_type_from_id_local)

# Condition of interest: mod1 says completion=True but reviewer completion=False.
mod1_fp_mask = (~work_df["reviewer_complete"].astype(bool)) & (work_df["qwen35_mod1_complete"].astype(bool))
MOD1_COMPLETION_FP_NODES = work_df.loc[mod1_fp_mask].copy()

if MOD1_COMPLETION_FP_NODES.empty:
    display(Markdown("### Mod1 Completion False Positives (Reviewer-Missed): None found in reviewed nodes."))
else:
    def _short_scenario_name_local(s: object) -> str:
        sc = str(s)
        if sc in SHORT_SCENARIO_MAP:
            return str(SHORT_SCENARIO_MAP[sc])
        txt = sc.split("/")[-1]
        txt = txt.replace("_GR1ArmsAndWaistFourierHands_Env", "")
        txt = txt.replace("SplitA", "")
        txt = txt.replace("Posttrain", "")
        txt = txt.replace("PnP", "")
        txt = txt.replace("_", " ").strip()
        return txt if txt else sc

    MOD1_COMPLETION_FP_NODES["short_scenario"] = MOD1_COMPLETION_FP_NODES["scenario_id"].map(_short_scenario_name_local)

    # 1) Node-type ranking where this mismatch happens most.
    fp_by_type = (
        MOD1_COMPLETION_FP_NODES.groupby("node_type", dropna=False)
        .agg(
            fp_nodes_n=("node_id", "size"),
            unique_node_ids=("node_id", "nunique"),
            affected_episodes_n=("episode_dir", "nunique"),
            affected_scenarios_n=("scenario_id", "nunique"),
        )
        .reset_index()
    )

    total_by_type = (
        work_df.groupby("node_type", dropna=False)
        .size()
        .rename("all_reviewed_nodes_n")
        .reset_index()
    )
    fp_by_type = fp_by_type.merge(total_by_type, on="node_type", how="left")
    fp_by_type["fp_rate_within_type_pct"] = (
        100.0 * fp_by_type["fp_nodes_n"] / fp_by_type["all_reviewed_nodes_n"]
    ).round(1)
    MOD1_FP_NODETYPE_SUMMARY = fp_by_type.sort_values(
        ["fp_nodes_n", "fp_rate_within_type_pct", "affected_episodes_n"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    display(Markdown("### Node-Type Ranking: Mod1 Marked Complete, Reviewer Did Not"))
    display(MOD1_FP_NODETYPE_SUMMARY)

    # 2) Focus on the most frequent node-type.
    top_node_type = str(MOD1_FP_NODETYPE_SUMMARY.iloc[0]["node_type"]).strip()
    TOP_FP_NODE_TYPE = top_node_type
    top_type_nodes = MOD1_COMPLETION_FP_NODES[MOD1_COMPLETION_FP_NODES["node_type"].astype(str) == top_node_type].copy()

    display(Markdown(
        f"### Top Node-Type for Mod1 Completion False Positives: **{top_node_type}**"
    ))

    # Episode-level rollup for this node-type.
    ep_rollup = (
        top_type_nodes.groupby(["episode_dir", "short_scenario", "scenario_id"], dropna=False)
        .agg(
            fp_node_count=("node_id", "size"),
            fp_unique_nodes=("node_id", "nunique"),
            node_indices=("node_index", lambda s: sorted(set(int(v) for v in s if pd.notna(v)))),
            node_ids=("node_id", lambda s: sorted(set(str(v) for v in s if pd.notna(v)))),
            reviewer_marked_failure_any=("reviewer_failed", "any"),
            mod1_marked_failure_any=("qwen35_mod1_failed", "any"),
            sim_success_any=("sim_success_bool", "any"),
        )
        .reset_index()
    )

    # Add total required nodes from base reviewed frame for context.
    nodes_required = (
        work_df.groupby("episode_dir", dropna=False)["node_index"]
        .max()
        .rename("required_action_nodes")
        .reset_index()
    )
    ep_rollup = ep_rollup.merge(nodes_required, on="episode_dir", how="left")
    ep_rollup["fp_share_of_required_nodes_pct"] = (
        100.0 * ep_rollup["fp_node_count"] / ep_rollup["required_action_nodes"].replace(0, np.nan)
    ).round(1)

    # 3) Representative 4-5 episode sample: prioritize diversity across scenario, then severity.
    ep_rollup = ep_rollup.sort_values(
        ["fp_node_count", "fp_unique_nodes", "fp_share_of_required_nodes_pct"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    scenario_sample = (
        ep_rollup.sort_values(["fp_node_count", "fp_unique_nodes"], ascending=[False, False])
        .drop_duplicates(subset=["short_scenario"], keep="first")
        .head(5)
    )

    sample_eps = scenario_sample.copy()
    if len(sample_eps) < 4:
        need = 4 - len(sample_eps)
        filler = ep_rollup[~ep_rollup["episode_dir"].isin(sample_eps["episode_dir"])].head(max(need, 0))
        sample_eps = pd.concat([sample_eps, filler], ignore_index=True)

    if len(sample_eps) < 5:
        filler = ep_rollup[~ep_rollup["episode_dir"].isin(sample_eps["episode_dir"])].head(5 - len(sample_eps))
        sample_eps = pd.concat([sample_eps, filler], ignore_index=True)

    MOD1_FP_TOPTYPE_EPISODE_SAMPLE = sample_eps.head(5).copy()
    MOD1_FP_TOPTYPE_EPISODE_SAMPLE = MOD1_FP_TOPTYPE_EPISODE_SAMPLE[[
        "short_scenario",
        "episode_dir",
        "required_action_nodes",
        "fp_node_count",
        "fp_unique_nodes",
        "fp_share_of_required_nodes_pct",
        "node_indices",
        "node_ids",
        "reviewer_marked_failure_any",
        "mod1_marked_failure_any",
        "sim_success_any",
    ]].reset_index(drop=True)

    display(Markdown("### Representative Episode Sample (4-5 episodes) from Top Node-Type"))
    display(MOD1_FP_TOPTYPE_EPISODE_SAMPLE)

    # Optional detailed node rows for these sampled episodes.
    sampled_episode_dirs = set(MOD1_FP_TOPTYPE_EPISODE_SAMPLE["episode_dir"].astype(str).tolist())
    MOD1_FP_TOPTYPE_NODE_DETAILS = (
        top_type_nodes[top_type_nodes["episode_dir"].astype(str).isin(sampled_episode_dirs)]
        [[
            "short_scenario",
            "episode_dir",
            "node_index",
            "node_id",
            "node_type",
            "reviewer_complete",
            "qwen35_mod1_complete",
            "reviewer_failed",
            "qwen35_mod1_failed",
            "reviewer_completion_s",
            "qwen35_mod1_completion_s",
            "sim_success_bool",
        ]]
        .sort_values(["short_scenario", "episode_dir", "node_index"])
        .reset_index(drop=True)
    )

    display(Markdown("### Node-Level Details for Sampled Episodes"))
    display(MOD1_FP_TOPTYPE_NODE_DETAILS)

## Mod1 Sample Episodes: Action Frame Inspection

For each sampled episode below, this section shows:
- node_description
- reviewer_complete
- qwen35_mo1_complete (mod1 completion flag)
- 4 representative frames for that action (first, two in-between, last)

Frame lookup is aligned to missionengine logs for the selected episode and action node.

In [ ]:
import json
import math
import re
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

if "MOD1_FP_TOPTYPE_NODE_DETAILS" not in globals() or MOD1_FP_TOPTYPE_NODE_DETAILS.empty:
    raise ValueError("MOD1_FP_TOPTYPE_NODE_DETAILS is missing or empty. Run the mod1 deep-dive sample cell first.")
if "MOD1_FP_TOPTYPE_EPISODE_SAMPLE" not in globals() or MOD1_FP_TOPTYPE_EPISODE_SAMPLE.empty:
    raise ValueError("MOD1_FP_TOPTYPE_EPISODE_SAMPLE is missing or empty. Run the mod1 deep-dive sample cell first.")
if "LOGS_DIR" not in globals():
    raise ValueError("LOGS_DIR is missing. Run the setup cell first.")

def _safe_text(v: object) -> str:
    if pd.isna(v):
        return ""
    return str(v)

def _extract_status_from_response_obj(obj: dict) -> str:
    if isinstance(obj.get("response"), dict) and isinstance(obj["response"].get("status"), str):
        return obj["response"].get("status", "")
    txt = _safe_text(obj.get("response_text", ""))
    if txt:
        m = re.search(r'"status"\s*:\s*"(running|complete|failure)"', txt)
        if m:
            return m.group(1)
    return ""

def _parse_node_description_from_prompt(prompt_text: str) -> str:
    m = re.search(r"Node description:\s*(.*?)\nSuccess criteria:", prompt_text, flags=re.DOTALL)
    if m:
        return m.group(1).strip()
    return ""

def _load_jsonl_records(jsonl_path: Path) -> list[dict]:
    recs = []
    try:
        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                try:
                    recs.append(json.loads(s))
                except Exception:
                    continue
    except Exception:
        return []
    return recs

def _resolve_episode_root(episode_dir: str) -> Path | None:
    ep = Path(episode_dir)
    if ep.is_absolute() and ep.exists():
        return ep
    cand = Path(LOGS_DIR) / episode_dir
    if cand.exists():
        return cand
    base = ep.name
    matches = [p for p in Path(LOGS_DIR).glob(f"**/{base}") if p.is_dir()]
    return matches[0] if matches else None

def _candidate_mission_files(ep_root: Path) -> list[Path]:
    if ep_root is None or not ep_root.exists():
        return []
    return sorted(ep_root.glob("**/missionengine.jsonl"))

def _score_mission_file_for_action(
    mission_path: Path,
    node_id: str,
    target_complete_step: float,
    preferred_run_hint: str = "",
) -> tuple[float, int, int, int]:
    recs = _load_jsonl_records(mission_path)
    req_steps = []
    complete_steps = []
    best_frame_window = 0

    for r in recs:
        if _safe_text(r.get("node_id", "")) != node_id:
            continue
        direction = _safe_text(r.get("direction", ""))
        step = r.get("step", np.nan)
        try:
            step_val = float(step)
        except Exception:
            step_val = np.nan

        if direction == "request":
            frames = r.get("frames", [])
            if isinstance(frames, list):
                best_frame_window = max(best_frame_window, len(frames))
            if np.isfinite(step_val):
                req_steps.append(step_val)

        if direction == "response":
            status = _extract_status_from_response_obj(r).lower()
            if status == "complete" and np.isfinite(step_val):
                complete_steps.append(step_val)

    best_dist = np.inf
    if np.isfinite(target_complete_step):
        if complete_steps:
            best_dist = min(abs(s - target_complete_step) for s in complete_steps)
        elif req_steps:
            best_dist = min(abs(s - target_complete_step) for s in req_steps)

    path_txt = str(mission_path).lower()
    hint_bonus = 0
    if preferred_run_hint and preferred_run_hint.lower() in path_txt:
        hint_bonus = 1

    # Sort by: closeness to target step, larger frame windows, more complete hits, more requests.
    return (best_dist, -best_frame_window - hint_bonus, -len(complete_steps), -len(req_steps))

def _choose_best_mission_file(
    ep_root: Path,
    node_id: str,
    target_complete_step: float,
    preferred_run_hint: str = "qwen_3_5",
) -> Path | None:
    cands = _candidate_mission_files(ep_root)
    if not cands:
        return None
    scored = [(
        _score_mission_file_for_action(p, node_id=node_id, target_complete_step=target_complete_step, preferred_run_hint=preferred_run_hint),
        p,
    ) for p in cands]
    scored.sort(key=lambda x: x[0])
    return scored[0][1]

def _select_action_record(
    mission_path: Path,
    node_id: str,
    target_complete_step: float,
    model_complete: bool,
) -> dict | None:
    recs = _load_jsonl_records(mission_path)
    best = None
    # Prefer fuller frame windows first, then step closeness, then response-status match.
    best_key = (1, np.inf, 1, np.inf)  # sparse_frames_flag, step_dist, status_penalty, index

    for i, r in enumerate(recs):
        if _safe_text(r.get("direction", "")) != "request":
            continue
        if _safe_text(r.get("node_id", "")) != node_id:
            continue
        frames = r.get("frames", [])
        if not isinstance(frames, list) or len(frames) == 0:
            continue

        frame_len = len(frames)
        sparse_flag = 1 if frame_len < 8 else 0

        step = r.get("step", np.nan)
        try:
            step_val = float(step)
        except Exception:
            step_val = np.nan
        step_dist = abs(step_val - target_complete_step) if np.isfinite(step_val) and np.isfinite(target_complete_step) else np.inf

        expected_status = "complete" if bool(model_complete) else "running"
        status_match_penalty = 0
        sid = _safe_text(r.get("status_eval_id", ""))
        resp_obj = None
        for rr in recs:
            if _safe_text(rr.get("direction", "")) == "response" and _safe_text(rr.get("status_eval_id", "")) == sid:
                resp_obj = rr
                break
        if resp_obj is not None:
            status = _extract_status_from_response_obj(resp_obj).lower()
            if status and status != expected_status:
                status_match_penalty = 1

        key = (sparse_flag, step_dist, status_match_penalty, i - frame_len * 0.01)
        if key < best_key:
            best_key = key
            best = r

    return best

def _extract_frame_index_from_name(p: str) -> int | None:
    m = re.search(r"(\d+)", Path(p).stem)
    if m:
        try:
            return int(m.group(1))
        except Exception:
            return None
    return None

def _episode_frame_pool(ep_root: Path) -> list[str]:
    if ep_root is None:
        return []
    ep_frames_dir = ep_root / "frames"
    if not ep_frames_dir.exists():
        return []
    imgs = list(ep_frames_dir.glob("*.jpg")) + list(ep_frames_dir.glob("*.png")) + list(ep_frames_dir.glob("*.jpeg"))
    imgs = sorted(imgs, key=lambda p: (_extract_frame_index_from_name(str(p)) is None, _extract_frame_index_from_name(str(p)) or 10**9, str(p)))
    return [str(p) for p in imgs]

def _pick_four_frames(frame_paths: list[str]) -> list[str]:
    valid = [p for p in frame_paths if isinstance(p, str) and p.strip() != "" and Path(p).exists()]
    if len(valid) == 0:
        return []
    if len(valid) <= 4:
        return valid
    idxs = sorted(set(int(round(x)) for x in np.linspace(0, len(valid) - 1, 4)))
    while len(idxs) < 4:
        for i in range(len(valid)):
            if i not in idxs:
                idxs.append(i)
                if len(idxs) == 4:
                    break
    idxs = sorted(idxs[:4])
    return [valid[i] for i in idxs]

def _augment_to_four(action_frames: list[str], ep_root: Path) -> list[str]:
    sel = _pick_four_frames(action_frames)
    if len(sel) >= 4:
        return sel

    pool = _episode_frame_pool(ep_root)
    if len(pool) == 0:
        return sel

    # If we have action frames, expand around their numeric span in episode pool.
    idx_vals = [_extract_frame_index_from_name(p) for p in action_frames]
    idx_vals = [v for v in idx_vals if v is not None]
    if idx_vals:
        lo, hi = min(idx_vals), max(idx_vals)
        window = [p for p in pool if (_extract_frame_index_from_name(p) is not None and lo <= _extract_frame_index_from_name(p) <= hi)]
        if len(window) >= 4:
            return _pick_four_frames(window)

    # Fallback to full-episode evenly spaced frames.
    return _pick_four_frames(pool)

def _node_description_from_episode_row(episode_dir: str, node_index: int) -> str:
    if "reviewed_eps" in globals() and isinstance(reviewed_eps, pd.DataFrame) and not reviewed_eps.empty:
        g = reviewed_eps[reviewed_eps["episode_dir"].astype(str) == str(episode_dir)]
        if not g.empty:
            col = f"node_{int(node_index)}_node_description"
            if col in g.columns:
                txt = _safe_text(g.iloc[0][col]).strip()
                if txt:
                    return txt
    return ""

# Build distinct episode/action rows for display (max 5).
base_rows = (
    MOD1_FP_TOPTYPE_NODE_DETAILS.sort_values(["short_scenario", "episode_dir", "node_index"])
    .drop_duplicates(subset=["episode_dir"], keep="first")
    .head(5)
    .reset_index(drop=True)
)

FRAME_REVIEW_SUMMARY_ROWS = []

for _, row in base_rows.iterrows():
    episode_dir = _safe_text(row.get("episode_dir", ""))
    short_scenario = _safe_text(row.get("short_scenario", ""))
    node_index = int(row.get("node_index", 0)) if pd.notna(row.get("node_index", np.nan)) else 0
    node_id = _safe_text(row.get("node_id", ""))
    reviewer_complete = bool(row.get("reviewer_complete", False))
    qwen35_mod1_complete = bool(row.get("qwen35_mod1_complete", False))
    target_step = float(row.get("qwen35_mod1_completion_s", np.nan)) if pd.notna(row.get("qwen35_mod1_completion_s", np.nan)) else np.nan

    ep_root = _resolve_episode_root(episode_dir)
    mission_path = _choose_best_mission_file(
        ep_root=ep_root,
        node_id=node_id,
        target_complete_step=target_step,
        preferred_run_hint="qwen_3_5",
    )

    action_req = _select_action_record(
        mission_path=mission_path,
        node_id=node_id,
        target_complete_step=target_step,
        model_complete=qwen35_mod1_complete,
    ) if mission_path is not None else None

    action_frames = []
    node_description = _node_description_from_episode_row(episode_dir, node_index)

    if action_req is not None:
        raw_frames = action_req.get("frames", [])
        if isinstance(raw_frames, list):
            action_frames = [str(p) for p in raw_frames]
        if not node_description:
            prompt_text = _safe_text(action_req.get("request", {}).get("prompt_text", ""))
            node_description = _parse_node_description_from_prompt(prompt_text)

    selected = _augment_to_four(action_frames=action_frames, ep_root=ep_root)

    display(Markdown(
        f"### Episode: `{episode_dir}`  \\\n"
        f"- short_scenario: **{short_scenario}**  \\\n"
        f"- node_id: **{node_id}** (node_index={node_index})  \\\n"
        f"- node_description: **{node_description if node_description else '[not found]'}**  \\\n"
        f"- reviewer_complete: **{reviewer_complete}**  \\\n"
        f"- qwen35_mo1_complete: **{qwen35_mod1_complete}**  \\\n"
        f"- missionengine_source: **{str(mission_path) if mission_path is not None else '[not found]'}**"
    ))

    if len(selected) == 0:
        display(Markdown("No frames could be resolved for this episode/action."))
    else:
        fig, axes = plt.subplots(1, len(selected), figsize=(4.0 * len(selected), 3.6))
        if len(selected) == 1:
            axes = [axes]
        labels = ["first", "in-between", "in-between", "last"][: len(selected)]

        for ax, img_path, lbl in zip(axes, selected, labels):
            try:
                img = mpimg.imread(img_path)
                ax.imshow(img)
                ax.set_title(f"{lbl}\n{Path(img_path).name}", fontsize=9)
                ax.axis("off")
            except Exception:
                ax.text(0.5, 0.5, f"Unreadable image\n{img_path}", ha="center", va="center", fontsize=8)
                ax.axis("off")
        plt.tight_layout()
        plt.show()

    FRAME_REVIEW_SUMMARY_ROWS.append({
        "episode_dir": episode_dir,
        "short_scenario": short_scenario,
        "node_id": node_id,
        "node_index": node_index,
        "node_description": node_description,
        "reviewer_complete": reviewer_complete,
        "qwen35_mo1_complete": qwen35_mod1_complete,
        "selected_frame_count": len(selected),
        "missionengine_source": str(mission_path) if mission_path is not None else "",
    })

MOD1_FP_FRAME_REVIEW_SUMMARY = pd.DataFrame(FRAME_REVIEW_SUMMARY_ROWS)
display(Markdown("### Frame Review Summary Table"))
display(MOD1_FP_FRAME_REVIEW_SUMMARY)

## Additional 5 Sample Episodes
Show the next five distinct episodes using the same frame gallery layout (first/in-between/in-between/last).

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
import numpy as np

required_symbols = [
    "MOD1_FP_TOPTYPE_NODE_DETAILS",
    "_safe_text",
    "_resolve_episode_root",
    "_choose_best_mission_file",
    "_select_action_record",
    "_node_description_from_episode_row",
    "_parse_node_description_from_prompt",
    "_augment_to_four",
    "mpimg",
    "plt",
    "Path",
]
missing = [s for s in required_symbols if s not in globals()]
if missing:
    raise ValueError(
        "Missing prerequisites from the previous frame review cell: " + ", ".join(missing)
    )

already_shown = set()
if "MOD1_FP_FRAME_REVIEW_SUMMARY" in globals() and isinstance(MOD1_FP_FRAME_REVIEW_SUMMARY, pd.DataFrame) and not MOD1_FP_FRAME_REVIEW_SUMMARY.empty:
    already_shown = set(MOD1_FP_FRAME_REVIEW_SUMMARY["episode_dir"].astype(str).tolist())

candidate_rows = (
    MOD1_FP_TOPTYPE_NODE_DETAILS.sort_values(["short_scenario", "episode_dir", "node_index"])
    .drop_duplicates(subset=["episode_dir"], keep="first")
    .reset_index(drop=True)
)

if already_shown:
    candidate_rows = candidate_rows[~candidate_rows["episode_dir"].astype(str).isin(already_shown)].reset_index(drop=True)

base_rows_next5 = candidate_rows.head(5).copy()

display(Markdown("### Additional Frame Review (Next 5 Episodes)"))
if base_rows_next5.empty:
    display(Markdown("No additional distinct episodes are available beyond the already displayed set."))
else:
    NEXT5_FRAME_REVIEW_SUMMARY_ROWS = []

    for _, row in base_rows_next5.iterrows():
        episode_dir = _safe_text(row.get("episode_dir", ""))
        short_scenario = _safe_text(row.get("short_scenario", ""))
        node_index = int(row.get("node_index", 0)) if pd.notna(row.get("node_index", np.nan)) else 0
        node_id = _safe_text(row.get("node_id", ""))
        reviewer_complete = bool(row.get("reviewer_complete", False))
        qwen35_mod1_complete = bool(row.get("qwen35_mod1_complete", False))
        target_step = float(row.get("qwen35_mod1_completion_s", np.nan)) if pd.notna(row.get("qwen35_mod1_completion_s", np.nan)) else np.nan

        ep_root = _resolve_episode_root(episode_dir)
        mission_path = _choose_best_mission_file(
            ep_root=ep_root,
            node_id=node_id,
            target_complete_step=target_step,
            preferred_run_hint="qwen_3_5",
        )

        action_req = _select_action_record(
            mission_path=mission_path,
            node_id=node_id,
            target_complete_step=target_step,
            model_complete=qwen35_mod1_complete,
        ) if mission_path is not None else None

        action_frames = []
        node_description = _node_description_from_episode_row(episode_dir, node_index)

        if action_req is not None:
            raw_frames = action_req.get("frames", [])
            if isinstance(raw_frames, list):
                action_frames = [str(p) for p in raw_frames]
            if not node_description:
                prompt_text = _safe_text(action_req.get("request", {}).get("prompt_text", ""))
                node_description = _parse_node_description_from_prompt(prompt_text)

        selected = _augment_to_four(action_frames=action_frames, ep_root=ep_root)

        display(Markdown(
            f"### Episode: `{episode_dir}`  \\\n"
            f"- short_scenario: **{short_scenario}**  \\\n"
            f"- node_id: **{node_id}** (node_index={node_index})  \\\n"
            f"- node_description: **{node_description if node_description else '[not found]'}**  \\\n"
            f"- reviewer_complete: **{reviewer_complete}**  \\\n"
            f"- qwen35_mo1_complete: **{qwen35_mod1_complete}**  \\\n"
            f"- missionengine_source: **{str(mission_path) if mission_path is not None else '[not found]'}**"
        ))

        if len(selected) == 0:
            display(Markdown("No frames could be resolved for this episode/action."))
        else:
            fig, axes = plt.subplots(1, len(selected), figsize=(4.0 * len(selected), 3.6))
            if len(selected) == 1:
                axes = [axes]
            labels = ["first", "in-between", "in-between", "last"][: len(selected)]

            for ax, img_path, lbl in zip(axes, selected, labels):
                try:
                    img = mpimg.imread(img_path)
                    ax.imshow(img)
                    ax.set_title(f"{lbl}\n{Path(img_path).name}", fontsize=9)
                    ax.axis("off")
                except Exception:
                    ax.text(0.5, 0.5, f"Unreadable image\n{img_path}", ha="center", va="center", fontsize=8)
                    ax.axis("off")
            plt.tight_layout()
            plt.show()

        NEXT5_FRAME_REVIEW_SUMMARY_ROWS.append({
            "episode_dir": episode_dir,
            "short_scenario": short_scenario,
            "node_id": node_id,
            "node_index": node_index,
            "node_description": node_description,
            "reviewer_complete": reviewer_complete,
            "qwen35_mo1_complete": qwen35_mod1_complete,
            "selected_frame_count": len(selected),
            "missionengine_source": str(mission_path) if mission_path is not None else "",
        })

    MOD1_FP_FRAME_REVIEW_SUMMARY_NEXT5 = pd.DataFrame(NEXT5_FRAME_REVIEW_SUMMARY_ROWS)
    display(Markdown("### Additional Frame Review Summary Table"))
    display(MOD1_FP_FRAME_REVIEW_SUMMARY_NEXT5)